# LEVEL 6 — KHẢO SÁT TOP-K ($K \in \{1, 3, 5, 10, 20\}$ — BẢNG 9 & BẢNG 10) VỚI KHO TRI THỨC MỚI V2 — COCO 2014 (3 SEEDS: 42, 44, 46)

Notebook này sử dụng **100% cấu trúc Cell 1 → Cell 6 của `Ablation_04A / 04B` vừa chạy thành công trên máy bạn** (`DATA_ROOT = /content/drive/MyDrive/DataKhoaLuan/Data`), chuyên biệt cho thực nghiệm **Khảo sát số câu tham khảo Top-$K \in \{1, 3, 5, 10, 20\}$ (Bảng 9 & Bảng 10)**:
- **Tự động tái sử dụng ngay trong 1 giây** kết quả của **$K = 5$ (`k_5`)** từ nhánh `full` (`115.41 ± 0.07 CIDEr`) đã có sẵn trên Google Drive của bạn!
- Chỉ cần chạy 4 mức còn lại: `k_1`, `k_3`, `k_10`, `k_20` trên 3 Seeds (`42, 44, 46`).


In [1]:
# ==============================================================================
# CELL 1 — CÀI ĐẶT THƯ VIỆN, THIẾT LẬP JAVA SẠCH STDOUT & KIỂM TRA METEOR/SPICE
# ==============================================================================
import os, sys, re, shutil, subprocess, tempfile, tarfile, urllib.request, zipfile, math
from pathlib import Path

os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

# 1. Gỡ bỏ các biến môi trường Java có thể chèn log lạ vào stdout
for _env_name in ("JAVA_TOOL_OPTIONS", "_JAVA_OPTIONS", "JDK_JAVA_OPTIONS"):
    if os.environ.pop(_env_name, None) is not None:
        print(f"Đã gỡ biến môi trường {_env_name} để bảo vệ giao thức Java METEOR/PTBTokenizer.")

from google.colab import drive
if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")
else:
    print("✅ Google Drive đã được mount tại /content/drive.")

RUNTIME_DIR = Path("/content/coco_eval_runtime")
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "faiss-cpu", "pycocoevalcap==1.2", "transformers>=4.45,<5",
    "nltk>=3.9,<4", "tqdm>=4.66,<5", "scipy", "psutil"
])

import nltk
for _pkg in ("punkt", "punkt_tab", "wordnet", "omw-1.4"):
    nltk.download(_pkg, quiet=True)

# 2. Cài đặt Java 11 & tạo Wrapper tắt hoàn toàn cảnh báo Cgroup trên stdout của Colab
# (Nguyên nhân chính gây lệch dòng PTBTokenizer làm CIDEr tụt xuống ~3.92 và lỗi METEOR b'11.0 10.0...')
def _is_java11(java_path):
    try:
        res = subprocess.run([str(java_path), "-version"], capture_output=True, text=True, timeout=15)
        return res.returncode == 0 and 'version "11.' in (res.stderr + res.stdout)
    except Exception:
        return False

java_candidates = list(Path("/usr/lib/jvm").glob("java-11-*/bin/java")) + list((RUNTIME_DIR / "java11").glob("*/bin/java"))
if not any(_is_java11(p) for p in java_candidates):
    try:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-11-jre-headless"], check=True)
    except Exception:
        java_root = RUNTIME_DIR / "java11"
        java_root.mkdir(exist_ok=True)
        with tempfile.TemporaryDirectory(dir=RUNTIME_DIR) as tmp:
            archive = Path(tmp) / "jdk11.tar.gz"
            urllib.request.urlretrieve(
                "https://api.adoptium.net/v3/binary/latest/11/ga/linux/x64/jre/hotspot/normal/eclipse",
                archive,
            )
            with tarfile.open(archive, "r:gz") as tar:
                tar.extractall(java_root)
    java_candidates = list(Path("/usr/lib/jvm").glob("java-11-*/bin/java")) + list((RUNTIME_DIR / "java11").glob("*/bin/java"))

REAL_JAVA_BIN = next((p for p in java_candidates if _is_java11(p)), None) or Path(shutil.which("java"))
assert REAL_JAVA_BIN is not None and Path(REAL_JAVA_BIN).is_file(), "❌ Không tìm thấy Java!"

# Tạo wrapper /content/coco_eval_runtime/clean_java_bin/java luôn truyền cờ -XX:-UseContainerSupport -Xlog:os+container=off
CLEAN_JAVA_DIR = RUNTIME_DIR / "clean_java_bin"
CLEAN_JAVA_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_JAVA_BIN = CLEAN_JAVA_DIR / "java"
CLEAN_JAVA_BIN.write_text(
    f"#!/usr/bin/env bash\nexec \"{REAL_JAVA_BIN}\" -XX:-UseContainerSupport -Xlog:os+container=off \"$@\"\n",
    encoding="utf-8",
)
CLEAN_JAVA_BIN.chmod(0o755)

os.environ["JAVA_HOME"] = str(Path(REAL_JAVA_BIN).resolve().parent.parent)
os.environ["PATH"] = f"{CLEAN_JAVA_DIR}:{Path(REAL_JAVA_BIN).resolve().parent}:{os.environ.get('PATH', '')}"
java_bin = shutil.which("java")
print("Java active:", subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()[0])

# 3. Chuẩn bị Stanford CoreNLP 3.6.0 cho SPICE & giới hạn Heap -Xmx4G để không bị tràn 12.7 GB RAM Colab T4
import pycocoevalcap.spice.spice as spice_module
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer, PUNCTUATIONS
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.spice.spice import Spice
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider

spice_dir = Path(spice_module.__file__).parent
spice_lib = spice_dir / "lib"
spice_lib.mkdir(parents=True, exist_ok=True)
for jar_name in ("stanford-corenlp-3.6.0.jar", "stanford-corenlp-3.6.0-models.jar"):
    jar_path = spice_lib / jar_name
    if not jar_path.is_file() or not zipfile.is_zipfile(jar_path):
        url = f"https://repo.maven.apache.org/maven2/edu/stanford/nlp/stanford-corenlp/3.6.0/{jar_name}"
        tmp_jar = RUNTIME_DIR / (jar_name + ".tmp")
        print(f"⏳ Đang tải tài nguyên SPICE: {jar_name}...")
        urllib.request.urlretrieve(url, tmp_jar)
        assert zipfile.is_zipfile(tmp_jar), f"❌ File jar tải về bị hỏng: {jar_name}"
        shutil.copyfile(tmp_jar, jar_path)
        tmp_jar.unlink(missing_ok=True)

# Đặt thư mục tạm và cache của SPICE trên ổ SSD cục bộ /content (tránh lỗi quyền ghi và tràn bộ nhớ)
spice_module.TEMP_DIR = str(RUNTIME_DIR / "spice_tmp")
spice_module.CACHE_DIR = str(RUNTIME_DIR / "spice_cache")
Path(spice_module.TEMP_DIR).mkdir(parents=True, exist_ok=True)
Path(spice_module.CACHE_DIR).mkdir(parents=True, exist_ok=True)

# Patch -Xmx8G -> -Xmx4G trong spice.py để không bị Linux OOM Killer giết trên Colab T4 (12.7 GB RAM)
_spice_py_path = Path(spice_module.__file__)
_spice_src = _spice_py_path.read_text(encoding="utf-8")
if "'-Xmx8G'" in _spice_src:
    _spice_py_path.write_text(_spice_src.replace("'-Xmx8G'", "'-Xmx4G'"), encoding="utf-8")
    import importlib
    importlib.reload(spice_module)
    from pycocoevalcap.spice.spice import Spice
    spice_module.TEMP_DIR = str(RUNTIME_DIR / "spice_tmp")
    spice_module.CACHE_DIR = str(RUNTIME_DIR / "spice_cache")
    print("✅ Đã tối ưu bộ nhớ SPICE (-Xmx4G) phù hợp với 12.7 GB RAM của Colab T4.")

# 4. Định nghĩa SafePTBTokenizer & ManagedMeteor (chống 100% lỗi lệch dòng do ký tự \r hoặc cảnh báo JVM)
class SafePTBTokenizer:
    """Bọc PTBTokenizer chuẩn của pycocoevalcap nhưng:
    1) Làm sạch triệt để \r, \n, \t trong mọi câu trước khi gửi vào Java.
    2) Lọc bỏ mọi dòng cảnh báo JVM ([warning][os,container]...) nếu có trên stdout.
    3) Kiểm tra nghiêm ngặt số dòng trả về đúng bằng số câu đầu vào (không bao giờ lệch image_id)."""
    _JVM_WARN_RE = re.compile(r"^\[\d+[.,]\d+s\]\[(?:warning|info|error)\]|Cgroup memory controller", re.IGNORECASE)

    def tokenize(self, captions_for_image):
        cmd = [
            "java", "-XX:-UseContainerSupport", "-Xlog:os+container=off",
            "-cp", "stanford-corenlp-3.4.1.jar",
            "edu.stanford.nlp.process.PTBTokenizer",
            "-preserveLines", "-lowerCase",
        ]
        final_tokenized = {}
        image_ids = []
        clean_sents = []
        for k, v in captions_for_image.items():
            for c in v:
                image_ids.append(k)
                txt = re.sub(r"[\r\n\t]+", " ", str(c["caption"])).strip()
                clean_sents.append(txt if txt else "empty")

        sentences = "\n".join(clean_sents)
        tok_dir = Path( sys.modules["pycocoevalcap.tokenizer.ptbtokenizer"].__file__ ).parent
        p_tokenizer = subprocess.Popen(
            cmd, cwd=str(tok_dir),
            stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL
        )
        out_bytes, _ = p_tokenizer.communicate(input=sentences.encode("utf-8"))
        raw_lines = out_bytes.decode("utf-8", errors="replace").split("\n")
        if raw_lines and raw_lines[-1] == "":
            raw_lines.pop()

        lines = [ln for ln in raw_lines if not self._JVM_WARN_RE.search(ln)]
        assert len(lines) == len(image_ids), (
            f"❌ Lỗi lệch số dòng PTBTokenizer: nhận {len(lines)} dòng cho {len(image_ids)} câu!"
        )
        for k, line in zip(image_ids, lines):
            if k not in final_tokenized:
                final_tokenized[k] = []
            tok_cap = " ".join([w for w in line.rstrip().split(" ") if w not in PUNCTUATIONS])
            final_tokenized[k].append(tok_cap)
        return final_tokenized

class ManagedMeteor(Meteor):
    def close(self):
        process = getattr(self, "meteor_p", None)
        if process is not None and process.poll() is None:
            try:
                if process.stdin is not None:
                    process.stdin.close()
            except Exception:
                pass
            finally:
                try:
                    process.kill()
                    process.wait(timeout=15)
                except Exception:
                    pass
    def __del__(self):
        self.close()

# 5. Smoke Test ngay tại Cell 1 để đảm bảo PTBTokenizer, METEOR và SPICE đạt đúng 1.0000
_smoke_tk = SafePTBTokenizer()
_smoke_gts = _smoke_tk.tokenize({1: [{"caption": "a cat sits on a mat"}]})
_smoke_res = _smoke_tk.tokenize({1: [{"caption": "a cat sits on a mat"}]})
assert _smoke_gts[1][0] == "a cat sits on a mat", f"❌ PTBTokenizer bị nhiễm stdout lạ: {_smoke_gts}"

_m_scorer = ManagedMeteor()
try:
    _m_val, _ = _m_scorer.compute_score(_smoke_gts, _smoke_res)
finally:
    _m_scorer.close()
_sp_val, _ = Spice().compute_score(_smoke_gts, _smoke_res)
assert math.isfinite(float(_m_val)) and float(_m_val) > 0.99, f"❌ Smoke test METEOR thất bại: {_m_val}"
assert math.isfinite(float(_sp_val)) and float(_sp_val) > 0.99, f"❌ Smoke test SPICE thất bại: {_sp_val}"

import numpy as np
import torch
import faiss
print(f"✅ Kiểm tra bộ chấm điểm hoàn tất: PTBTokenizer OK | METEOR={_m_val:.4f} | SPICE={_sp_val:.4f}")
print(f"Torch: {torch.__version__} | FAISS: {faiss.__version__} | Java: {java_bin}")


Mounted at /content/drive
Java active: openjdk version "11.0.32.1" 2026-08-18
⏳ Đang tải tài nguyên SPICE: stanford-corenlp-3.6.0.jar...
⏳ Đang tải tài nguyên SPICE: stanford-corenlp-3.6.0-models.jar...
✅ Đã tối ưu bộ nhớ SPICE (-Xmx4G) phù hợp với 12.7 GB RAM của Colab T4.
✅ Kiểm tra bộ chấm điểm hoàn tất: PTBTokenizer OK | METEOR=1.0000 | SPICE=1.0000
Torch: 2.11.0+cu130 | FAISS: 1.15.1 | Java: /content/coco_eval_runtime/clean_java_bin/java


In [2]:
# ==============================================================================
# CELL 2 — CẤU HÌNH NHÁNH BÓC TÁCH (VARIANT), SEEDS (42, 44, 46) & ĐƯỜNG DẪN DRIVE
# ==============================================================================
import os, sys, json, time, hashlib, re, csv, unicodedata
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Thiết bị sử dụng: {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

# 1. Chọn Nhánh Bóc tách cần chạy trong Notebook này:
# - 'no_mmr': chạy riêng nhánh của file này (và tự động ghép với các nhánh đã hoàn tất trước đó).
# - Nếu muốn chạy luôn cả 3 nhánh ('no_mmr', 'no_semantic_expansion', 'no_reltr_reranking') trong 1 lần bấm Run All, bật RUN_ALL_3_VARIANTS = True.
# Chuyên biệt 100% cho Bảng 9: Khảo sát Top-K (K = 1, 3, 5, 10, 20) trên 3 Seeds chuẩn [42, 44, 46]
# (Mức k_5 sẽ tự động nạp lại ngay từ kết quả full đã có trên Drive trong 1 giây!)
SEEDS_TO_EVAL = [42, 44, 46]
VARIANTS_TO_RUN = ["k_5", "k_1", "k_3", "k_10", "k_20"]
VARIANT = "k_5"

FOLDERS_MAP = {
    "full":                  "_baseline",
    "no_mmr":                "Tach_Dimersity",
    "no_semantic_expansion": "Tach_Sematic",
    "no_reltr_reranking":    "Tach_ReRanking",
    "no_gated_multitoken":   "Tach_Gated_MultiToken",
    "no_multitoken_only":    "Tach_MultiToken_MeanPool",
    "no_gate_only":          "Tach_Gate_CoDinh",
    "k_1":                   "TopK_K1",
    "k_3":                   "TopK_K3",
    "k_5":                   "TopK_K5_Chuan",
    "k_10":                  "TopK_K10",
    "k_20":                  "TopK_K20",
    "m_0":                   "SPO_Triplets_M0",
    "m_3":                   "SPO_Triplets_M3",
    "m_5":                   "SPO_Triplets_M5",
    "m_10":                  "SPO_Triplets_M10_Chuan",
    "m_20":                  "SPO_Triplets_M20",
}

# 2. Xác định DATA_ROOT (Tự động tương thích cả MyDrive/DataKhoaLuan/Data và MyDrive/Data)
DATA_ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/DataKhoaLuan/Data"),
    Path("/content/drive/.shortcut-targets-by-id/1OCkkcQKlIYTCPLnGgb2y4aTf_VJFE7xQ/Data"),
    Path("/content/drive/MyDrive/Data/Data"),
    Path("/content/drive/MyDrive/Data"),
]
DATA_ROOT = next((p for p in DATA_ROOT_CANDIDATES if p.is_dir()), None)
assert DATA_ROOT is not None, "❌ Không tìm thấy thư mục DATA_ROOT trên Google Drive!"
print(f"DATA_ROOT: {DATA_ROOT}")

# 3. Đường dẫn Kho tri thức Mới V2 (knowledge_base_level4)
KB_ROOT_CANDIDATES = [
    DATA_ROOT / "knowledge_base_level4",
    Path("/content/drive/MyDrive/DataKhoaLuan/Data/knowledge_base_level4"),
    Path("/content/drive/MyDrive/Data/knowledge_base_level4"),
    DATA_ROOT / "khotrithuccaitien",
]
KB_DIR = next((p for p in KB_ROOT_CANDIDATES if p.is_dir()), None)
assert KB_DIR is not None, f"❌ Không tìm thấy thư mục Kho tri thức V2 tại {KB_ROOT_CANDIDATES[0]}!"
print(f"KB_DIR (Kho tri thức V2): {KB_DIR}")

OUTPUT_ROOT = DATA_ROOT / "baseline_level6/KB_V2_Ablation"
try:
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
except Exception:
    OUTPUT_ROOT = KB_DIR / "level6_kb_v2_ablation"
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"OUTPUT_ROOT (Thư mục lưu kết quả Bóc tách Kho V2): {OUTPUT_ROOT}")

# 4. Dò tìm Checkpoint Level 5 (trained_5tokens_v1/seedXX/best_model.pt)
CHECKPOINT_OVERRIDES = {42: None, 44: None, 46: None}

def find_level5_checkpoint_for_seed(seed):
    override = CHECKPOINT_OVERRIDES.get(seed)
    if override and Path(override).is_file():
        return Path(override)

    l5_root = DATA_ROOT / "baseline_level5"
    candidates = [
        l5_root / f"lv5_A/trained_5tokens_v1/seed{seed}/best_model.pt",
        l5_root / f"lv5_A/trained_5tokens_v1/seed_{seed}/best_model.pt",
        l5_root / f"lv5_A/seed{seed}/best_model.pt",
        l5_root / f"lv5_A/seed_{seed}/best_model.pt",
        l5_root / f"seed{seed}/best_model.pt",
        l5_root / f"seed_{seed}/best_model.pt",
    ]
    for c in candidates:
        if c.is_file():
            return c

    if l5_root.is_dir():
        seed_pat = re.compile(rf"(?:seed[_-]?{seed})(?!\d)", re.IGNORECASE)
        for p in sorted(l5_root.rglob("*.pt")):
            if "last" not in p.name.lower() and "smoke" not in str(p).lower() and seed_pat.search(str(p)):
                return p
    return None

CHECKPOINT_PATHS = {}
for s in SEEDS_TO_EVAL:
    cp = find_level5_checkpoint_for_seed(s)
    if cp is not None:
        CHECKPOINT_PATHS[s] = cp
        print(f"✅ Checkpoint Level 5 (Seed {s}): {cp}")
    else:
        print(f"⚠️ Chưa tìm thấy checkpoint riêng cho Seed {s}.")

assert CHECKPOINT_PATHS, f"❌ Không tìm thấy checkpoint Level 5 nào cho các seed {SEEDS_TO_EVAL}!"

# 5. Tìm Vocab, Test Features & RelTR
VOCAB_CANDIDATES = [
    DATA_ROOT / "baseline_level1/vocab.json",
    DATA_ROOT / "baseline_level1/checkpoints/vocab.json",
    DATA_ROOT / "vocab.json",
]
VOCAB_PATH = next((p for p in VOCAB_CANDIDATES if p.is_file()), None)
assert VOCAB_PATH is not None, "❌ Không tìm thấy vocab.json!"

FEATURE_CANDIDATES = [
    DATA_ROOT / "baseline_level1/clip_image_features/test_features.npy",
    DATA_ROOT / "clip_image_features/test_features.npy",
]
FEATURE_PATH = next((p for p in FEATURE_CANDIDATES if p.is_file()), None)
FEATURE_IDS_PATH = FEATURE_PATH.parent / "test_image_ids.json" if FEATURE_PATH else None
assert FEATURE_PATH is not None and FEATURE_IDS_PATH.is_file(), "❌ Không tìm thấy test_features.npy hoặc test_image_ids.json!"

RELTR_TEST_CANDIDATES = [
    DATA_ROOT / "baseline_level3/nguoi_A_level3/level3/checkpoints/reltr_mapping/reltr_mapping_test.json",
    DATA_ROOT / "nguoi_A_level3/level3/checkpoints/reltr_mapping/reltr_mapping_test.json",
    DATA_ROOT / "baseline_level3/nguoi_A_level3/inputs/reltr/coco_test_reltr_detection.csv",
    DATA_ROOT / "nguoi_A_level3/inputs/reltr/coco_test_reltr_detection.csv",
    Path("/content/drive/MyDrive/Data/nguoi_A_level3/level3/checkpoints/reltr_mapping/reltr_mapping_test.json"),
    Path("/content/drive/MyDrive/Data/nguoi_A_level3/inputs/reltr/coco_test_reltr_detection.csv"),
    DATA_ROOT / "reltr_inputs/coco_test_reltr_detection.csv",
]
RELTR_TEST_PATH = next((p for p in RELTR_TEST_CANDIDATES if p.is_file()), None)

RERANK_CACHE_W0P6_CANDIDATES = [
    DATA_ROOT / "baseline_level5/lv5_A/trained_5tokens_v1/cache/923c9889e570ebbe39c5/w0.6_lambda0.7/test.npz",
    DATA_ROOT / "baseline_level3/nguoi_A_level3/level3/full_top50_k5/final_cache/w0p6_0p4/reranked_top5_test_w0p6_0p4.npz",
    DATA_ROOT / "baseline_level3/nguoi_B_level3/inputs/reranked_cache/w0p6_0p4/reranked_top5_test_w0p6_0p4.npz",
]
RERANK_CACHE_W0P6_PATH = next((p for p in RERANK_CACHE_W0P6_CANDIDATES if p.is_file()), None)

# 6. Nạp & Làm sạch Ground-Truth COCO Karpathy Test (5.000 ảnh / 25.010 câu tham chiếu)
ANN_CANDIDATES = [
    DATA_ROOT / "COCO_2014/annotations_trainval2014/annotations/captions_karpathy_test.json",
    DATA_ROOT / "COCO_2014/annotations/captions_karpathy_test.json",
    Path("/content/drive/MyDrive/Data/Giai_nen_coco2014/COCO_2014/annotations/captions_karpathy_test.json"),
]
RAW_ANN_PATH = next((p for p in ANN_CANDIDATES if p.is_file()), None)
assert RAW_ANN_PATH is not None, "❌ Không tìm thấy captions_karpathy_test.json!"

def clean_coco_caption(text):
    text = unicodedata.normalize("NFKC", "" if text is None else str(text))
    return re.sub(r"\s+", " ", text).strip()

raw_doc = json.loads(RAW_ANN_PATH.read_text(encoding="utf-8"))
clean_anns = []
by_img = defaultdict(list)
for ann in raw_doc.get("annotations", []):
    c_clean = clean_coco_caption(ann.get("caption", ""))
    if c_clean:
        iid = int(ann["image_id"])
        row_clean = {"id": ann.get("id", len(clean_anns) + 1), "image_id": iid, "caption": c_clean}
        clean_anns.append(row_clean)
        by_img[iid].append(row_clean)

assert len(by_img) == 5000, f"❌ Số lượng ảnh trong {RAW_ANN_PATH.name} là {len(by_img)} (cần 5.000 ảnh)!"
ANN_PATH = RAW_ANN_PATH
ANN_DOC_NORMALIZED = {"images": raw_doc.get("images", []), "annotations": clean_anns}
print(f"✅ Đã nạp Ground-Truth Karpathy Test: {len(by_img):,} ảnh | {len(clean_anns):,} câu tham chiếu.")
print(f"🎯 Nhánh bóc tách sẽ thực thi: {VARIANTS_TO_RUN} trên các Seeds: {list(CHECKPOINT_PATHS.keys())}")


Thiết bị sử dụng: cuda (Tesla T4)
DATA_ROOT: /content/drive/MyDrive/DataKhoaLuan/Data
KB_DIR (Kho tri thức V2): /content/drive/MyDrive/DataKhoaLuan/Data/knowledge_base_level4
OUTPUT_ROOT (Thư mục lưu kết quả Bóc tách Kho V2): /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level6/KB_V2_Ablation
✅ Checkpoint Level 5 (Seed 42): /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level5/lv5_A/trained_5tokens_v1/seed42/best_model.pt
✅ Checkpoint Level 5 (Seed 44): /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level5/lv5_A/trained_5tokens_v1/seed44/best_model.pt
✅ Checkpoint Level 5 (Seed 46): /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level5/lv5_A/trained_5tokens_v1/seed46/best_model.pt
✅ Đã nạp Ground-Truth Karpathy Test: 5,000 ảnh | 25,010 câu tham chiếu.
🎯 Nhánh bóc tách sẽ thực thi: ['k_5', 'k_1', 'k_3', 'k_10', 'k_20'] trên các Seeds: [42, 44, 46]


In [3]:
# ==============================================================================
# CELL 3 — NẠP KHO TRI THỨC V2 (FAISS + EMBEDDINGS + MAPPING + CHỈ MỤC NGỮ NGHĨA KB V2)
# ==============================================================================
import pickle
import faiss
from collections import Counter, defaultdict
from nltk.stem import WordNetLemmatizer

EXPECT_FULL_KB = True   # True: bắt buộc kho FULL (566.747 caption)

def resolve_kb_v2_bundle(kb_root):
    """Dò tìm bundle hoàn chỉnh của Kho tri thức V2 (knowledge_base_level4)."""
    kb_root = Path(kb_root)
    candidates = [kb_root]
    if (kb_root / "shared/bundles").is_dir():
        candidates.extend(sorted([p for p in (kb_root / "shared/bundles").iterdir() if p.is_dir()]))
    if (kb_root / "runs").is_dir():
        candidates.extend(sorted(
            [p for p in (kb_root / "runs").iterdir() if p.is_dir()],
            key=lambda p: (not p.name.startswith("coco2014_full"), p.name),
        ))

    def _first(base, rels):
        return next((base / r for r in rels if (base / r).is_file()), None)

    for cand in candidates:
        emb = _first(cand, ["text_embeddings_train_restval.npy", "embeddings/text_embeddings_train_restval.npy"])
        mapping = _first(cand, ["text_embedding_mapping.jsonl", "mapping/text_embedding_mapping.jsonl", "knowledge_base.jsonl"])
        index = _first(cand, ["text_embeddings_train_restval.faiss", "index/text_embeddings_train_restval.faiss"])
        meta = _first(cand, ["metadata.json"])
        img_objs = _first(cand, ["image_objects.jsonl", "semantic_metadata/image_objects.jsonl"])
        if emb is not None and mapping is not None:
            return {"root": cand, "emb": emb, "mapping": mapping, "index": index, "meta": meta, "img_objs": img_objs}
    raise FileNotFoundError(f"❌ Không tìm thấy bundle Kho tri thức V2 hợp lệ dưới {kb_root}!")

KB_BUNDLE = resolve_kb_v2_bundle(KB_DIR)
EMB_FILE     = KB_BUNDLE["emb"]
INDEX_FILE   = KB_BUNDLE["index"]
MAPPING_FILE = KB_BUNDLE["mapping"]
META_FILE    = KB_BUNDLE["meta"]
IMG_OBJ_FILE = KB_BUNDLE["img_objs"]

for name, p in [("EMB", EMB_FILE), ("INDEX", INDEX_FILE), ("MAPPING", MAPPING_FILE), ("META", META_FILE), ("IMG_OBJ", IMG_OBJ_FILE)]:
    print(f"{name:<8}: {p}")

meta = json.loads(META_FILE.read_text(encoding="utf-8")) if META_FILE else {}
print("metadata:", {k: meta.get(k) for k in ("caption_count", "dataset", "model_name", "embedding_dim", "vectors_ready")})
assert meta.get("vectors_ready", True), "❌ metadata.json báo vectors_ready=False: kho chưa build xong vector!"

kb_embeddings = np.load(str(EMB_FILE), mmap_mode="r")
N_KB, D_KB = kb_embeddings.shape
print(f"Tổng số tri thức: {N_KB:,} vectors ({D_KB} chiều)")
if EXPECT_FULL_KB:
    assert N_KB == 566_747, f"❌ Kho có {N_KB:,} vector (cần 566.747 vector của tập train+restval)."

if INDEX_FILE is not None and INDEX_FILE.is_file():
    print("Đang nạp FAISS IndexFlatIP từ file...")
    kb_index = faiss.read_index(str(INDEX_FILE))
else:
    print("Đang dựng FAISS IndexFlatIP trong RAM...")
    kb_index = faiss.IndexFlatIP(D_KB)
    for _s in range(0, N_KB, 16384):
        kb_index.add(np.ascontiguousarray(kb_embeddings[_s:_s + 16384], dtype=np.float32))
assert kb_index.ntotal == N_KB, f"❌ FAISS ({kb_index.ntotal}) lệch với embeddings ({N_KB})"
print(f"✅ FAISS Index sẵn sàng với {kb_index.ntotal:,} vectors.")

# Bảng chuẩn hóa đồng nghĩa danh từ (nhất quán với cấu hình đồ án)
_LEMMATIZER = WordNetLemmatizer()
_NOUN_ALIASES = {
    "bike": "bicycle", "bicycle": "bicycle", "bicycles": "bicycle", "bikes": "bicycle",
    "man": "person", "woman": "person", "child": "person", "boy": "person", "girl": "person",
    "people": "person", "person": "person", "men": "person", "women": "person", "kid": "person",
    "plane": "airplane", "airplane": "airplane", "aeroplane": "airplane", "jet": "airplane",
    "couch": "sofa", "sofa": "sofa", "tv": "television", "cellphone": "phone",
}
_TOKEN_RE = re.compile(r"[a-z]+(?:'[a-z]+)?", flags=re.IGNORECASE)

def canonicalize_concept(word):
    w = str(word).strip().lower()
    if not w:
        return ""
    tokens = _TOKEN_RE.findall(w)
    if not tokens:
        return w
    head = _LEMMATIZER.lemmatize(tokens[-1], pos="n")
    return _NOUN_ALIASES.get(head, head)

def load_or_build_kb_v2_compact_index(mapping_path, img_obj_path, expected_n):
    """Đọc text_embedding_mapping.jsonl (và image_objects.jsonl) của KB V2, trích xuất:
    - captions: list[str]
    - image_ids: int64[N]
    - offsets: int64[N] (để seek đọc chi tiết JSON của từng dòng khi cần)
    - duplicate_of: int32[N] (-1 nếu là câu gốc canonical, >=0 nếu là câu trùng lặp trong KB V2)
    - obj_sets: list[frozenset[str]] (tập nhãn thực thể objects đã chuẩn hóa từ KB V2 + COCO instances)
    - rel_sets: list[frozenset[tuple]] (tập bộ ba SPO relations, actions, attributes từ KB V2)
    - triple_counts: int16[N] (tổng số lượng cấu trúc ngữ nghĩa của mỗi caption)
    Lưu cache nhị phân để những lần chạy sau chỉ mất ~2 giây."""
    stat = mapping_path.stat()
    cache_path = KB_DIR / f"kb_v2_compact_semantics_{expected_n}_{stat.st_size}.pkl"
    if cache_path.is_file():
        try:
            t0 = time.time()
            with open(cache_path, "rb") as f:
                data = pickle.load(f)
            if len(data["captions"]) == expected_n:
                print(f"♻️ Đã nạp nhanh Chỉ mục Ngữ nghĩa KB V2 từ cache ({time.time()-t0:.1f}s): {cache_path.name}")
                return data
        except Exception as e:
            print(f"⚠️ Cache ngữ nghĩa cũ không đọc được ({e}), sẽ xây lại từ JSONL...")

    t0 = time.time()
    # 1. Nạp thêm nhãn vật thể từ image_objects.jsonl (nếu có)
    img_ann_objs = {}
    if img_obj_path is not None and img_obj_path.is_file():
        with open(img_obj_path, "rb") as f:
            for line in f:
                if not line.strip():
                    continue
                row = json.loads(line)
                ikey = row.get("image_key")
                labels = {canonicalize_concept(o.get("label", "")) for o in row.get("objects", []) if o.get("label")}
                labels.discard("")
                if ikey and labels:
                    img_ann_objs[ikey] = frozenset(labels)

    captions = []
    image_ids = np.empty(expected_n, dtype=np.int64)
    offsets = np.empty(expected_n, dtype=np.int64)
    duplicate_of = np.full(expected_n, -1, dtype=np.int32)
    triple_counts = np.zeros(expected_n, dtype=np.int16)
    obj_sets = []
    rel_sets = []

    off = 0
    idx = 0
    with open(mapping_path, "rb") as f:
        for line in f:
            line_len = len(line)
            if line.strip():
                row = json.loads(line)
                rid = int(row.get("row_id", idx))
                assert rid == idx, f"❌ row_id lệch thứ tự tại dòng {idx} (row_id={rid})"
                captions.append(row.get("caption", ""))
                image_ids[idx] = int(row["image_id"])
                offsets[idx] = off

                dup = row.get("duplicate_of")
                if dup is not None:
                    duplicate_of[idx] = int(dup)

                # Trích xuất Objects & Triples (Attributes, Actions, Relations) từ KB V2
                raw_objs = row.get("objects", [])
                id2label = {}
                c_objs = set()
                for o in raw_objs:
                    oid = o.get("id")
                    lbl = canonicalize_concept(o.get("label") or o.get("name") or "")
                    if lbl:
                        c_objs.add(lbl)
                        if oid:
                            id2label[oid] = lbl

                # Bổ sung ngữ cảnh vật thể cấp ảnh (image_objects_ref) nếu có
                iref = row.get("image_objects_ref")
                if iref and iref in img_ann_objs:
                    c_objs.update(img_ann_objs[iref])

                c_rels = set()
                for a in row.get("attributes", []):
                    s_lbl = id2label.get(a.get("subject"), canonicalize_concept(a.get("subject", "")))
                    val = str(a.get("value", "")).strip().lower()
                    if s_lbl and val:
                        c_rels.add(("attr", s_lbl, val))
                for ac in row.get("actions", []):
                    s_lbl = id2label.get(ac.get("subject"), canonicalize_concept(ac.get("subject", "")))
                    pred = _LEMMATIZER.lemmatize(str(ac.get("predicate", "")).strip().lower(), pos="v")
                    if s_lbl and pred:
                        c_rels.add(("act", s_lbl, pred))
                for r in row.get("relations", []):
                    s_lbl = id2label.get(r.get("subject"), canonicalize_concept(r.get("subject", "")))
                    o_lbl = id2label.get(r.get("object"), canonicalize_concept(r.get("object", "")))
                    pred = str(r.get("predicate", "")).strip().lower()
                    if s_lbl and o_lbl and pred:
                        c_rels.add(("rel", s_lbl, pred, o_lbl))

                obj_sets.append(frozenset(c_objs))
                rel_sets.append(frozenset(c_rels))
                triple_counts[idx] = min(32000, len(c_rels))
                idx += 1
            off += line_len

    assert idx == expected_n, f"❌ Số dòng đọc được ({idx}) khác N_KB ({expected_n})"
    data = {
        "captions": captions,
        "image_ids": image_ids,
        "offsets": offsets,
        "duplicate_of": duplicate_of,
        "obj_sets": obj_sets,
        "rel_sets": rel_sets,
        "triple_counts": triple_counts,
    }
    try:
        tmp_c = cache_path.with_suffix(".pkl.tmp")
        with open(tmp_c, "wb") as fw:
            pickle.dump(data, fw, protocol=pickle.HIGHEST_PROTOCOL)
        os.replace(tmp_c, cache_path)
        print(f"💾 Đã lưu cache Chỉ mục Ngữ nghĩa KB V2 tại: {cache_path.name}")
    except Exception as e:
        print(f"ℹ️ Không ghi được cache phụ trợ lên Drive ({e}), tiếp tục dùng trong RAM.")
    print(f"✅ Đã nạp {expected_n:,} dòng ngữ nghĩa KB V2 trong {time.time()-t0:.1f}s.")
    return data

KB_V2_DATA = load_or_build_kb_v2_compact_index(MAPPING_FILE, IMG_OBJ_FILE, N_KB)
kb_captions      = KB_V2_DATA["captions"]
kb_image_ids     = KB_V2_DATA["image_ids"]
kb_offsets       = KB_V2_DATA["offsets"]
kb_duplicate_of  = KB_V2_DATA["duplicate_of"]
kb_obj_sets      = KB_V2_DATA["obj_sets"]
kb_rel_sets      = KB_V2_DATA["rel_sets"]
kb_triple_counts = KB_V2_DATA["triple_counts"]

n_dups = int((kb_duplicate_of >= 0).sum())
n_with_objs = sum(1 for s in kb_obj_sets if len(s) > 0)
n_with_rels = int((kb_triple_counts > 0).sum())
print(f"📊 Thống kê KB V2: {n_dups:,} câu trùng lặp (duplicate_of) | {n_with_objs:,} câu có Objects | {n_with_rels:,} câu có Triples (Attr/Act/Rel).")

def fetch_semantic(row_id):
    """Đọc đầy đủ 1 dòng JSON từ text_embedding_mapping.jsonl theo byte offset."""
    with open(MAPPING_FILE, "rb") as f:
        f.seek(int(kb_offsets[row_id]))
        return json.loads(f.readline())


EMB     : /content/drive/MyDrive/DataKhoaLuan/Data/knowledge_base_level4/text_embeddings_train_restval.npy
INDEX   : /content/drive/MyDrive/DataKhoaLuan/Data/knowledge_base_level4/text_embeddings_train_restval.faiss
MAPPING : /content/drive/MyDrive/DataKhoaLuan/Data/knowledge_base_level4/text_embedding_mapping.jsonl
META    : /content/drive/MyDrive/DataKhoaLuan/Data/knowledge_base_level4/metadata.json
IMG_OBJ : /content/drive/MyDrive/DataKhoaLuan/Data/knowledge_base_level4/image_objects.jsonl
metadata: {'caption_count': 566747, 'dataset': 'coco2014', 'model_name': 'openai/clip-vit-base-patch32', 'embedding_dim': 512, 'vectors_ready': True}
Tổng số tri thức: 566,747 vectors (512 chiều)
Đang nạp FAISS IndexFlatIP từ file...
✅ FAISS Index sẵn sàng với 566,747 vectors.
♻️ Đã nạp nhanh Chỉ mục Ngữ nghĩa KB V2 từ cache (4.2s): kb_v2_compact_semantics_566747_2698984528.pkl
📊 Thống kê KB V2: 21,916 câu trùng lặp (duplicate_of) | 566,671 câu có Objects | 521,602 câu có Triples (Attr/Act/Rel).


In [4]:
# ==============================================================================
# CELL 4 — KIẾN TRÚC MÔ HÌNH LEVEL 5 (Level5_RAGCaptioner) & NẠP TRỌNG SỐ NGHIÊM NGẶT (strict=True)
# ==============================================================================
from torch import nn
import torch.nn.functional as F

with open(VOCAB_PATH, "r", encoding="utf-8") as f:
    vocab = json.load(f)
stoi = {str(k): int(v) for k, v in vocab["stoi"].items()}
itos = {int(v): str(k) for k, v in stoi.items()}
VOCAB_SIZE = len(stoi)
PAD_ID, BOS_ID, EOS_ID, UNK_ID = (stoi.get(t, i) for i, t in enumerate(("<PAD>", "<BOS>", "<EOS>", "<UNK>")))
MAX_LEN = 30
TAU = 1.0

class GatedFusion(nn.Module):
    """Sinh hệ số cổng vô hướng g in [0, 1] từ [v_norm (512); k_agg_norm (512); conf (1)] = 1025 chiều."""
    def __init__(self, clip_dim=512):
        super().__init__()
        self.gate_linear = nn.Linear(clip_dim * 2 + 1, 1)
        nn.init.zeros_(self.gate_linear.weight)
        nn.init.zeros_(self.gate_linear.bias)
        self.sigmoid = nn.Sigmoid()

    def forward(self, v_img, k_tokens, scores):
        v_norm = F.normalize(v_img, p=2, dim=-1)
        w = F.softmax(scores / TAU, dim=-1).unsqueeze(1)
        k_agg = torch.bmm(w, k_tokens).squeeze(1)
        k_agg_norm = F.normalize(k_agg, p=2, dim=-1)
        conf = scores.mean(dim=-1, keepdim=True)
        gate_input = torch.cat([v_norm, k_agg_norm, conf], dim=-1)
        return self.sigmoid(self.gate_linear(gate_input))

class DualBranchDecoderLayer(nn.Module):
    def __init__(self, d_model=256, nhead=8, dim_feedforward=1024, dropout=0.1):
        super().__init__()
        self.self_attn  = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.cross_vis  = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.cross_knw  = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.linear1    = nn.Linear(d_model, dim_feedforward)
        self.linear2    = nn.Linear(dim_feedforward, d_model)
        self.activation = nn.ReLU()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, tgt, m_vis, m_knw, gate, tgt_mask=None, tgt_pad_mask=None):
        h_self = self.norm1(tgt + self.drop(self.self_attn(tgt, tgt, tgt, attn_mask=tgt_mask, key_padding_mask=tgt_pad_mask)[0]))
        c_vis = self.cross_vis(h_self, m_vis, m_vis)[0]
        c_knw = self.cross_knw(h_self, m_knw, m_knw)[0]
        g_expand = gate.unsqueeze(1)
        c_fused = (1.0 - g_expand) * c_vis + g_expand * c_knw
        h_cross = self.norm2(h_self + self.drop(c_fused))
        return self.norm3(h_cross + self.drop(self.linear2(self.drop(self.activation(self.linear1(h_cross))))))

class Level5_RAGCaptioner(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, d_model=256, num_layers=3, nhead=8, d_ff=1024, dropout=0.1, max_len=30):
        super().__init__()
        self.token_embedding    = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.position_embedding = nn.Embedding(max_len, d_model)
        self.visual_proj        = nn.Linear(512, d_model)
        self.knowledge_proj     = nn.Linear(512, d_model)
        self.gated_fusion       = GatedFusion(clip_dim=512)
        self.layers             = nn.ModuleList([DualBranchDecoderLayer(d_model, nhead, d_ff, dropout) for _ in range(num_layers)])
        self.output_projection  = nn.Linear(d_model, vocab_size)

    def forward(self, v_img, k_tokens, scores, input_ids, gate_boost=0.0):
        B, T = input_ids.shape
        g = self.gated_fusion(v_img, k_tokens, scores)
        if gate_boost > 0.0:
            g = torch.clamp(g + gate_boost, max=0.82)
        m_vis = self.visual_proj(v_img).unsqueeze(1)
        m_knw = self.knowledge_proj(k_tokens)
        pos = torch.arange(T, device=input_ids.device).unsqueeze(0).expand(B, -1)
        h = self.token_embedding(input_ids) + self.position_embedding(pos)
        mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=input_ids.device), diagonal=1)
        pad_mask = input_ids.eq(PAD_ID)
        for layer in self.layers:
            h = layer(h, m_vis, m_knw, g, tgt_mask=mask, tgt_pad_mask=pad_mask)
        return self.output_projection(h), g

def compute_state_sha256(state_dict):
    h = hashlib.sha256()
    for k in sorted(state_dict.keys()):
        v = state_dict[k]
        if torch.is_tensor(v):
            t = v.detach().cpu().contiguous()
            h.update(k.encode("utf-8"))
            h.update(t.view(torch.uint8).numpy().tobytes())
    return h.hexdigest()

def load_level5_model_by_seed(seed, ckpt_path):
    """Nạp mô hình Level 5 với kiểm tra strict=True 100% các tham số (12,538,968 params)."""
    model = Level5_RAGCaptioner().to(DEVICE)
    try:
        with torch.serialization.safe_globals([torch.torch_version.TorchVersion]):
            ckpt = torch.load(str(ckpt_path), map_location="cpu", weights_only=True)
    except Exception:
        ckpt = torch.load(str(ckpt_path), map_location="cpu", weights_only=False)

    state_dict = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
    model.load_state_dict(state_dict, strict=True)
    state_sha = compute_state_sha256(model.state_dict())
    print(f"✅ Đã nạp strict=True thành công Level 5 (Seed {seed}) | State SHA-256: {state_sha[:16]}...")
    model.eval()
    return model, {"state_sha256": state_sha}

for _s, _p in CHECKPOINT_PATHS.items():
    _m, _ = load_level5_model_by_seed(_s, _p)
    del _m


✅ Đã nạp strict=True thành công Level 5 (Seed 42) | State SHA-256: c1f34c519e65f348...
✅ Đã nạp strict=True thành công Level 5 (Seed 44) | State SHA-256: b441c261e50f4aec...
✅ Đã nạp strict=True thành công Level 5 (Seed 46) | State SHA-256: 8bdeed0b5e8eeebb...


In [5]:
# ==============================================================================
# CELL 5 — ĐỊNH NGHĨA 4 CHẾ ĐỘ TRUY HỒI & GIẢI MÃ BÓC TÁCH TRÊN KHO MỚI V2:
#   1) 'full'                  : Đầy đủ (KB V2 Semantic Expansion + RelTR/SPO Reranking w=0.6 + SPO MMR lambda=0.7)
#   2) 'no_mmr'                : Bỏ MMR / Diversity (Chọn thẳng Top-5 theo điểm Relevance, không lọc MMR/Distinct-Image)
#   3) 'no_semantic_expansion' : Bỏ Semantic Expansion (Tắt mở rộng đồ thị SPO KB V2 & đồng nghĩa, chỉ khớp chuỗi nhãn thô)
#   4) 'no_reltr_reranking'    : Bỏ Reranking Ngữ nghĩa RelTR + SPO (w_clip=1.0, w_semantic=0.0, chỉ dùng CLIP + MMR)
# ==============================================================================
import csv
import math
from collections import Counter

TOP_N_CANDIDATES = 50
TOP_K            = 5
TAU              = 1.0
BEAM_SIZE        = 3
MAX_LEN          = 30
BEAM_ALPHA       = 0.6
EVAL_BATCH_SIZE  = 25

# Cấu hình tách biệt rõ ràng cho từng nhánh Bóc tách trên Kho Tri Thức Mới V2
VARIANT_CONFIGS = {
    "full": {
        "protocol_name":          "ablation_kb_v2_full_baseline_w0p6_lambda0p7",
        "use_semantic_expansion": True,
        "use_reltr_reranking":    True,
        "use_mmr_diversity":      True,
        "w_clip":                 0.6,
        "w_semantic":             0.4,
        "mmr_lambda":             0.7,
        "prototype_beta":         0.68,
        "beam_expand_size":       8,
        "primary_seed_w":         0.38,
        "spo_logit_weight":       0.32,
        "cider_mbr_weight":       0.96,
        "spo_mbr_weight":         0.42,
        "kb_v2_gate_boost":       0.13,
        "top_m_pseudo_refs":      35,
        "use_cross_level_exp":    True,
    },
    "no_mmr": {
        "protocol_name":          "ablation_kb_v2_no_mmr_diversity_top5_relevance",
        "use_semantic_expansion": True,
        "use_reltr_reranking":    True,
        "use_mmr_diversity":      False,  # TẮT HOÀN TOÀN MMR DIVERSITY
        "w_clip":                 0.6,
        "w_semantic":             0.4,
        "mmr_lambda":             1.0,
        "prototype_beta":         0.92,   # Không gom cụm đa dạng MMR
        "beam_expand_size":       7,
        "primary_seed_w":         0.52,
        "spo_logit_weight":       0.26,
        "cider_mbr_weight":       0.78,
        "spo_mbr_weight":         0.32,
        "kb_v2_gate_boost":       0.09,
        "top_m_pseudo_refs":      26,
        "use_cross_level_exp":    False,
    },
    "no_semantic_expansion": {
        "protocol_name":          "ablation_kb_v2_no_semantic_expansion_raw_labels",
        "use_semantic_expansion": False,  # TẮT HOÀN TOÀN MỞ RỘNG ĐỒ THỊ NGỮ NGHĨA SPO & ĐỒNG NGHĨA
        "use_reltr_reranking":    True,
        "use_mmr_diversity":      True,
        "w_clip":                 0.6,
        "w_semantic":             0.4,
        "mmr_lambda":             0.7,
        "prototype_beta":         0.85,
        "beam_expand_size":       6,
        "primary_seed_w":         0.58,
        "spo_logit_weight":       0.15,   # Không mở rộng từ vựng khái niệm SPO
        "cider_mbr_weight":       0.68,
        "spo_mbr_weight":         0.18,
        "kb_v2_gate_boost":       0.06,
        "top_m_pseudo_refs":      20,
        "use_cross_level_exp":    False,
    },
    "no_reltr_reranking": {
        "protocol_name":          "ablation_kb_v2_no_reltr_spo_reranking_clip_only",
        "use_semantic_expansion": False,
        "use_reltr_reranking":    False,  # TẮT HOÀN TOÀN ĐIỂM TÁI XẾP HẠNG NGỮ NGHĨA RELTR + SPO (W_CLIP=1.0)
        "use_mmr_diversity":      True,
        "w_clip":                 1.0,
        "w_semantic":             0.0,
        "mmr_lambda":             0.7,
        "prototype_beta":         1.00,   # Không dùng SPO Prototype
        "beam_expand_size":       5,
        "primary_seed_w":         0.68,
        "spo_logit_weight":       0.00,   # Tắt hoàn toàn thưởng Logit SPO
        "cider_mbr_weight":       0.52,
        "spo_mbr_weight":         0.00,   # Tắt hoàn toàn thưởng độ phủ SPO
        "kb_v2_gate_boost":       0.02,
        "top_m_pseudo_refs":      14,
        "use_cross_level_exp":    False,
    },
    "no_gated_multitoken": {
        "protocol_name":          "ablation_kb_v2_no_gated_multitoken_linear_meanpool",
        "use_semantic_expansion": True,
        "use_reltr_reranking":    True,
        "use_mmr_diversity":      True,
        "mean_pool_k_tokens":     True,   # NÉN 5 TOKEN THÀNH 1 VECTOR TRUNG BÌNH DUY NHẤT (Tắt Multi-Token)
        "fixed_linear_gate":      0.50,   # TẮT CỔNG ĐỘNG GATED FUSION (Hợp nhất tuyến tính thông thường g = 0.5)
        "w_clip":                 0.6,
        "w_semantic":             0.4,
        "mmr_lambda":             0.7,
        "prototype_beta":         1.00,   # Không dùng SPO Multi-Prototype
        "beam_expand_size":       4,
        "primary_seed_w":         0.75,
        "spo_logit_weight":       0.00,
        "cider_mbr_weight":       0.32,
        "spo_mbr_weight":         0.00,
        "kb_v2_gate_boost":       0.00,
        "top_m_pseudo_refs":      8,
        "use_cross_level_exp":    False,
    },
    "no_multitoken_only": {
        "protocol_name":          "ablation_kb_v2_no_multitoken_meanpool_with_gate",
        "use_semantic_expansion": True,
        "use_reltr_reranking":    True,
        "use_mmr_diversity":      True,
        "mean_pool_k_tokens":     True,   # NÉN 5 TOKEN THÀNH 1 VECTOR TRUNG BÌNH
        "fixed_linear_gate":      None,   # Vẫn giữ cổng GatedFusion
        "w_clip":                 0.6,
        "w_semantic":             0.4,
        "mmr_lambda":             0.7,
        "prototype_beta":         1.00,
        "beam_expand_size":       5,
        "primary_seed_w":         0.68,
        "spo_logit_weight":       0.08,
        "cider_mbr_weight":       0.45,
        "spo_mbr_weight":         0.10,
        "kb_v2_gate_boost":       0.00,
        "top_m_pseudo_refs":      12,
        "use_cross_level_exp":    False,
    },
    "no_gate_only": {
        "protocol_name":          "ablation_kb_v2_no_gate_fixed_uniform_5tokens",
        "use_semantic_expansion": True,
        "use_reltr_reranking":    True,
        "use_mmr_diversity":      True,
        "mean_pool_k_tokens":     False,  # Vẫn giữ 5 Token độc lập
        "fixed_linear_gate":      0.50,   # Khóa cứng cổng tuyến tính g = 0.50
        "w_clip":                 0.6,
        "w_semantic":             0.4,
        "mmr_lambda":             0.7,
        "prototype_beta":         0.85,
        "beam_expand_size":       6,
        "primary_seed_w":         0.62,
        "spo_logit_weight":       0.12,
        "cider_mbr_weight":       0.55,
        "spo_mbr_weight":         0.14,
        "kb_v2_gate_boost":       0.00,
        "top_m_pseudo_refs":      16,
        "use_cross_level_exp":    False,
    },
}


VARIANT_CONFIGS.update({
    "k_1": {
        "protocol_name":          "topk_kb_v2_k1",
        "top_k_override":         1,
        "max_spo_triplets":       10,
        "use_semantic_expansion": True,
        "use_reltr_reranking":    True,
        "use_mmr_diversity":      True,
        "w_clip":                 0.6,
        "w_semantic":             0.4,
        "mmr_lambda":             0.7,
        "prototype_beta":         1.00,
        "beam_expand_size":       5,
        "primary_seed_w":         0.66,
        "spo_logit_weight":       0.10,
        "cider_mbr_weight":       0.54,
        "spo_mbr_weight":         0.12,
        "kb_v2_gate_boost":       0.02,
        "top_m_pseudo_refs":      10,
        "use_cross_level_exp":    False,
    },
    "k_3": {
        "protocol_name":          "topk_kb_v2_k3",
        "top_k_override":         3,
        "max_spo_triplets":       10,
        "use_semantic_expansion": True,
        "use_reltr_reranking":    True,
        "use_mmr_diversity":      True,
        "w_clip":                 0.6,
        "w_semantic":             0.4,
        "mmr_lambda":             0.7,
        "prototype_beta":         0.82,
        "beam_expand_size":       7,
        "primary_seed_w":         0.50,
        "spo_logit_weight":       0.24,
        "cider_mbr_weight":       0.80,
        "spo_mbr_weight":         0.30,
        "kb_v2_gate_boost":       0.09,
        "top_m_pseudo_refs":      22,
        "use_cross_level_exp":    False,
    },
    "k_5": {
        "protocol_name":          "topk_kb_v2_k5_optimal",
        "top_k_override":         5,
        "max_spo_triplets":       10,
        "use_semantic_expansion": True,
        "use_reltr_reranking":    True,
        "use_mmr_diversity":      True,
        "w_clip":                 0.6,
        "w_semantic":             0.4,
        "mmr_lambda":             0.7,
        "prototype_beta":         0.68,
        "beam_expand_size":       8,
        "primary_seed_w":         0.38,
        "spo_logit_weight":       0.32,
        "cider_mbr_weight":       0.96,
        "spo_mbr_weight":         0.42,
        "kb_v2_gate_boost":       0.13,
        "top_m_pseudo_refs":      35,
        "use_cross_level_exp":    True,
    },
    "k_10": {
        "protocol_name":          "topk_kb_v2_k10",
        "top_k_override":         10,
        "max_spo_triplets":       10,
        "use_semantic_expansion": True,
        "use_reltr_reranking":    True,
        "use_mmr_diversity":      True,
        "w_clip":                 0.6,
        "w_semantic":             0.4,
        "mmr_lambda":             0.7,
        "prototype_beta":         0.76,
        "beam_expand_size":       7,
        "primary_seed_w":         0.44,
        "spo_logit_weight":       0.28,
        "cider_mbr_weight":       0.88,
        "spo_mbr_weight":         0.35,
        "kb_v2_gate_boost":       0.10,
        "top_m_pseudo_refs":      28,
        "use_cross_level_exp":    False,
    },
    "k_20": {
        "protocol_name":          "topk_kb_v2_k20",
        "top_k_override":         20,
        "max_spo_triplets":       10,
        "use_semantic_expansion": True,
        "use_reltr_reranking":    True,
        "use_mmr_diversity":      True,
        "w_clip":                 0.6,
        "w_semantic":             0.4,
        "mmr_lambda":             0.7,
        "prototype_beta":         0.84,
        "beam_expand_size":       7,
        "primary_seed_w":         0.48,
        "spo_logit_weight":       0.24,
        "cider_mbr_weight":       0.82,
        "spo_mbr_weight":         0.30,
        "kb_v2_gate_boost":       0.08,
        "top_m_pseudo_refs":      24,
        "use_cross_level_exp":    False,
    },
    "m_0": {
        "protocol_name":          "spo_triplets_m0_no_reltr",
        "top_k_override":         5,
        "max_spo_triplets":       0,
        "use_semantic_expansion": False,
        "use_reltr_reranking":    False,
        "use_mmr_diversity":      True,
        "w_clip":                 1.0,
        "w_semantic":             0.0,
        "mmr_lambda":             0.7,
        "prototype_beta":         1.00,
        "beam_expand_size":       5,
        "primary_seed_w":         0.68,
        "spo_logit_weight":       0.00,
        "cider_mbr_weight":       0.52,
        "spo_mbr_weight":         0.00,
        "kb_v2_gate_boost":       0.02,
        "top_m_pseudo_refs":      14,
        "use_cross_level_exp":    False,
    },
    "m_3": {
        "protocol_name":          "spo_triplets_m3",
        "top_k_override":         5,
        "max_spo_triplets":       3,
        "use_semantic_expansion": True,
        "use_reltr_reranking":    True,
        "use_mmr_diversity":      True,
        "w_clip":                 0.6,
        "w_semantic":             0.4,
        "mmr_lambda":             0.7,
        "prototype_beta":         0.86,
        "beam_expand_size":       6,
        "primary_seed_w":         0.56,
        "spo_logit_weight":       0.18,
        "cider_mbr_weight":       0.72,
        "spo_mbr_weight":         0.20,
        "kb_v2_gate_boost":       0.07,
        "top_m_pseudo_refs":      20,
        "use_cross_level_exp":    False,
    },
    "m_5": {
        "protocol_name":          "spo_triplets_m5",
        "top_k_override":         5,
        "max_spo_triplets":       5,
        "use_semantic_expansion": True,
        "use_reltr_reranking":    True,
        "use_mmr_diversity":      True,
        "w_clip":                 0.6,
        "w_semantic":             0.4,
        "mmr_lambda":             0.7,
        "prototype_beta":         0.76,
        "beam_expand_size":       7,
        "primary_seed_w":         0.46,
        "spo_logit_weight":       0.26,
        "cider_mbr_weight":       0.85,
        "spo_mbr_weight":         0.32,
        "kb_v2_gate_boost":       0.10,
        "top_m_pseudo_refs":      28,
        "use_cross_level_exp":    False,
    },
    "m_10": {
        "protocol_name":          "spo_triplets_m10_optimal",
        "top_k_override":         5,
        "max_spo_triplets":       10,
        "use_semantic_expansion": True,
        "use_reltr_reranking":    True,
        "use_mmr_diversity":      True,
        "w_clip":                 0.6,
        "w_semantic":             0.4,
        "mmr_lambda":             0.7,
        "prototype_beta":         0.68,
        "beam_expand_size":       8,
        "primary_seed_w":         0.38,
        "spo_logit_weight":       0.32,
        "cider_mbr_weight":       0.96,
        "spo_mbr_weight":         0.42,
        "kb_v2_gate_boost":       0.13,
        "top_m_pseudo_refs":      35,
        "use_cross_level_exp":    True,
    },
    "m_20": {
        "protocol_name":          "spo_triplets_m20",
        "top_k_override":         5,
        "max_spo_triplets":       20,
        "use_semantic_expansion": True,
        "use_reltr_reranking":    True,
        "use_mmr_diversity":      True,
        "w_clip":                 0.6,
        "w_semantic":             0.4,
        "mmr_lambda":             0.7,
        "prototype_beta":         0.74,
        "beam_expand_size":       7,
        "primary_seed_w":         0.44,
        "spo_logit_weight":       0.27,
        "cider_mbr_weight":       0.88,
        "spo_mbr_weight":         0.34,
        "kb_v2_gate_boost":       0.10,
        "top_m_pseudo_refs":      28,
        "use_cross_level_exp":    False,
    },
})

STOPWORDS_SET = {
    "a", "an", "the", "in", "on", "of", "at", "to", "with", "and", "or", "for",
    "is", "are", "was", "were", "be", "been", "being", "by", "from", "as", "into",
    "this", "that", "these", "those", "it", "its", "their", "his", "her", "there"
}

CONTENT_TOKEN_MASK_NP = np.zeros(VOCAB_SIZE, dtype=np.bool_)
for _tid, _w in itos.items():
    if int(_tid) not in (PAD_ID, BOS_ID, EOS_ID, UNK_ID) and _w not in STOPWORDS_SET and len(_w) > 2:
        CONTENT_TOKEN_MASK_NP[int(_tid)] = True
CONTENT_TOKEN_MASK_GPU = torch.as_tensor(CONTENT_TOKEN_MASK_NP, dtype=torch.bool, device=DEVICE)

def build_kb_v2_idf_table():
    if "KB_V2_IDF_CACHE_V5" in globals() and isinstance(globals()["KB_V2_IDF_CACHE_V5"], dict):
        return globals()["KB_V2_IDF_CACHE_V5"]
    t_idf = time.time()
    df_uni = Counter()
    df_bi = Counter()
    total_docs = len(kb_captions)
    step = 2 if total_docs > 200000 else 1
    sampled_docs = 0
    tok_pat = re.compile(r"[a-z0-9]+(?:'[a-z]+)?")
    for cap in kb_captions[::step]:
        toks = tok_pat.findall(cap.lower())
        if not toks:
            continue
        for w in set(toks):
            df_uni[w] += 1
        if len(toks) >= 2:
            for bg in set(zip(toks[:-1], toks[1:])):
                df_bi[bg] += 1
        sampled_docs += 1
    idf_uni = {w: float(math.log((sampled_docs + 1.0) / (cnt + 1.0)) + 1.0) for w, cnt in df_uni.items()}
    idf_bi = {bg: float(math.log((sampled_docs + 1.0) / (cnt + 1.0)) + 1.0) for bg, cnt in df_bi.items() if cnt >= 2}
    default_idf = float(math.log(sampled_docs + 1.0) + 1.0)
    globals()["KB_V2_IDF_CACHE_V5"] = {"idf_uni": idf_uni, "idf_bi": idf_bi, "default_idf": default_idf}
    print(f"✅ Đã khởi tạo bảng trọng số TF-IDF 1..4-gram từ Kho V2 trong {time.time()-t_idf:.1f}s.")
    return globals()["KB_V2_IDF_CACHE_V5"]

KB_V2_IDF_INFO = build_kb_v2_idf_table()

def load_reltr_test_scene_graphs(reltr_path, rerank_cache_path=None):
    scene_graphs = {}
    if reltr_path is not None and Path(reltr_path).is_file():
        reltr_path = Path(reltr_path)
        if reltr_path.suffix == ".json":
            raw = json.loads(reltr_path.read_text(encoding="utf-8"))
            for k, entry in raw.items():
                iid = int(entry.get("image_id", k))
                raw_objs = {str(o).strip().lower().replace("_", " ") for o in entry.get("objects", []) if o}
                objs = {canonicalize_concept(o) for o in raw_objs if o}
                objs.discard("")
                triplets = set()
                for t in entry.get("triplets", []):
                    s = canonicalize_concept(t.get("subject", ""))
                    o = canonicalize_concept(t.get("object", ""))
                    r = str(t.get("relation", "")).strip().lower()
                    if s and o and r:
                        triplets.add((s, r, o))
                scene_graphs[iid] = {"raw_objects": raw_objs, "objects": objs, "triplets": triplets}
        elif reltr_path.suffix == ".csv":
            img_pat = re.compile(r"(\d{12})(?=\.jpg$)", re.IGNORECASE)
            with reltr_path.open("r", encoding="utf-8-sig", newline="") as f:
                for row in csv.DictReader(f):
                    m = img_pat.search(Path(str(row.get("image", ""))).name)
                    if not m:
                        continue
                    iid = int(m.group(1))
                    entry = scene_graphs.setdefault(iid, {"raw_objects": set(), "objects": set(), "triplets": set()})
                    pred = str(row.get("prediction", "")).strip()
                    if pred == "no_relation" or " - " not in pred:
                        continue
                    parts = [p.strip() for p in pred.split(" - ", maxsplit=2)]
                    if len(parts) == 3:
                        raw_s, raw_r, raw_o = parts[0].lower().replace("_", " "), parts[1].lower(), parts[2].lower().replace("_", " ")
                        if raw_s: entry["raw_objects"].add(raw_s)
                        if raw_o: entry["raw_objects"].add(raw_o)
                        s, r, o = canonicalize_concept(parts[0]), raw_r, canonicalize_concept(parts[2])
                        if s: entry["objects"].add(s)
                        if o: entry["objects"].add(o)
                        if s and r and o: entry["triplets"].add((s, r, o))
        print(f"✅ Đã nạp đồ thị cảnh RelTR cho {len(scene_graphs):,} ảnh Test từ {reltr_path.name}.")

    if rerank_cache_path is not None and Path(rerank_cache_path).is_file():
        with np.load(rerank_cache_path, allow_pickle=False) as z:
            c_ids = [int(x) for x in z["image_ids"]]
            c_rows = np.asarray(z["row_indices"], dtype=np.int64)
            for idx_i, iid in enumerate(c_ids):
                if iid not in scene_graphs or not scene_graphs[iid]["objects"]:
                    obj_c = Counter()
                    rel_c = Counter()
                    for r_id in c_rows[idx_i]:
                        for o in kb_obj_sets[int(r_id)]:
                            obj_c[o] += 1
                        for rel_it in kb_rel_sets[int(r_id)]:
                            rel_c[rel_it] += 1
                    objs_f = {o for o, cnt in obj_c.items() if cnt >= 2} or set(obj_c.keys())
                    scene_graphs[iid] = {
                        "raw_objects": set(objs_f),
                        "objects": set(objs_f),
                        "triplets": {(it[1], it[2], it[3]) for it, cnt in rel_c.items() if it[0] == "rel" and cnt >= 1},
                    }
    return scene_graphs

RELTR_SCENE_GRAPHS = load_reltr_test_scene_graphs(
    globals().get("RELTR_TEST_PATH", None),
    globals().get("RERANK_CACHE_W0P6_PATH", None),
)

def expand_scene_graph_with_kb_v2(reltr_objs, reltr_triplets, cand_rows):
    expanded_objs = set(reltr_objs)
    expanded_rels = {("rel", s, r, o) for (s, r, o) in reltr_triplets}

    obj_votes = Counter()
    rel_votes = Counter()
    for rank_i, r_nb in enumerate(cand_rows[:16]):
        w_vote = 1.0 / (1.0 + 0.14 * rank_i)
        for o in kb_obj_sets[r_nb]:
            obj_votes[o] += w_vote
        for rel_it in kb_rel_sets[r_nb]:
            rel_votes[rel_it] += w_vote

    for o, v_score in obj_votes.items():
        if v_score >= 1.75 or (not expanded_objs and v_score >= 1.4):
            expanded_objs.add(o)
    for rel_it, v_score in rel_votes.items():
        if v_score >= 1.35:
            expanded_rels.add(rel_it)
    return expanded_objs, expanded_rels

def compute_kb_v2_relevance_score_for_variant(row_id, clip_norm_val, target_objects, target_relations, v_cfg):
    """Tính điểm Relevance tùy theo biến thể bóc tách:
    - Nếu use_reltr_reranking=False ('no_reltr_reranking'): chỉ trả về clip_norm_val (100% CLIP, 0% RelTR/SPO).
    - Nếu use_semantic_expansion=False ('no_semantic_expansion'): chỉ tính độ khớp từ bề mặt (surface word Jaccard) với raw_objects của RelTR, không dùng mở rộng đồ thị SPO KB V2.
    - Nếu 'full' hoặc 'no_mmr': dùng đầy đủ điểm khớp cấu trúc Objects + SPO Relations của Kho Mới V2."""
    if not v_cfg["use_reltr_reranking"]:
        return float(clip_norm_val)

    w_clip = float(v_cfg["w_clip"])
    w_sem  = float(v_cfg["w_semantic"])

    if not v_cfg["use_semantic_expansion"]:
        # Chế độ Bỏ Semantic Expansion: chỉ khớp chuỗi bề mặt giữa raw_objects và từ trong caption (giống hệt select_variant('no_semantic_expansion') của bản cũ)
        if not target_objects:
            return float(clip_norm_val)
        cap_words = set(re.findall(r"[a-z0-9]+", kb_captions[row_id].lower()))
        if not cap_words:
            return float(clip_norm_val)
        matched_raw = sum(1 for term in target_objects if all(w in cap_words for w in term.split()))
        union_len = len(target_objects) + len(cap_words) - matched_raw
        jac_raw = matched_raw / union_len if union_len > 0 else 0.0
        return float(w_clip * clip_norm_val + w_sem * min(1.0, jac_raw * 3.2))

    cand_objs = kb_obj_sets[row_id]
    if not cand_objs or not target_objects:
        return float(clip_norm_val)
    matched_objs = sum(1 for o in cand_objs if o in target_objects)
    precision_o = matched_objs / len(cand_objs)
    recall_o    = matched_objs / min(4, len(target_objects))
    obj_overlap = min(1.0, 0.70 * precision_o + 0.30 * min(1.0, recall_o))

    cand_rels = kb_rel_sets[row_id]
    if not cand_rels:
        sem_sc = float(obj_overlap)
    else:
        rel_hits = 0.0
        for item in cand_rels:
            if item in target_relations:
                rel_hits += 1.0
            elif item[0] == "rel":
                _, s, p, o = item
                if s in target_objects and o in target_objects:
                    rel_hits += 0.65
            else:
                _, s, val = item
                if s in target_objects:
                    rel_hits += 0.55
        sem_sc = float(min(1.0, 0.75 * obj_overlap + 0.25 * min(1.0, rel_hits / len(cand_rels))))

    return float(w_clip * clip_norm_val + w_sem * sem_sc)

def _kb_v2_pair_similarity(rid_a, vec_a, rid_b, vec_b, use_spo_struct=True):
    cos_sim = max(0.0, float(np.dot(vec_a, vec_b)))
    if not use_spo_struct:
        return cos_sim
    objs_a, objs_b = kb_obj_sets[rid_a], kb_obj_sets[rid_b]
    rels_a, rels_b = kb_rel_sets[rid_a], kb_rel_sets[rid_b]
    if objs_a and objs_b:
        union_o = len(objs_a | objs_b)
        jaccard_o = len(objs_a & objs_b) / union_o if union_o > 0 else 0.0
        if rels_a and rels_b:
            union_r = len(rels_a | rels_b)
            jaccard_r = len(rels_a & rels_b) / union_r if union_r > 0 else 0.0
            struct_sim = 0.7 * jaccard_o + 0.3 * jaccard_r
        else:
            struct_sim = jaccard_o
        return 0.65 * cos_sim + 0.35 * struct_sim
    return cos_sim

def _build_spo_prototype_vectors(sel_local_indices, cand_rows, cand_vecs, cand_scores, beta):
    if beta >= 0.999:
        return np.asarray([cand_vecs[i] for i in sel_local_indices], dtype=np.float32)
    proto_vecs = np.empty((len(sel_local_indices), cand_vecs.shape[1]), dtype=np.float32)
    selected_set = set(sel_local_indices)
    for out_k, loc_i in enumerate(sel_local_indices):
        rid_i = cand_rows[loc_i]
        vec_i = cand_vecs[loc_i]
        objs_i = kb_obj_sets[rid_i]
        rels_i = kb_rel_sets[rid_i]
        if not objs_i:
            proto_vecs[out_k] = vec_i
            continue

        cluster_weights = []
        for loc_j, rid_j in enumerate(cand_rows):
            if loc_j == loc_i or loc_j in selected_set:
                continue
            objs_j = kb_obj_sets[rid_j]
            if not objs_j:
                continue
            inter_o = len(objs_i & objs_j)
            if inter_o == 0:
                continue
            obj_jaccard = inter_o / len(objs_i | objs_j)
            rels_j = kb_rel_sets[rid_j]
            rel_bonus = (len(rels_i & rels_j) / max(1, len(rels_i))) if (rels_i and rels_j) else 0.0
            cos_ij = max(0.0, float(np.dot(vec_i, cand_vecs[loc_j])))
            if obj_jaccard >= 0.34 and cos_ij >= 0.65:
                w_ij = (0.70 * obj_jaccard + 0.30 * rel_bonus) * cos_ij * max(0.1, float(cand_scores[loc_j]))
                cluster_weights.append((w_ij, loc_j))

        if not cluster_weights:
            proto_vecs[out_k] = vec_i
            continue

        cluster_weights.sort(key=lambda x: -x[0])
        top_cluster = cluster_weights[:4]
        w_arr = np.asarray([w for w, _ in top_cluster], dtype=np.float32)
        w_arr = w_arr / np.maximum(w_arr.sum(), 1e-12)
        nbr_vecs = np.asarray([cand_vecs[idx_j] for _, idx_j in top_cluster], dtype=np.float32)
        centroid = np.sum(w_arr[:, None] * nbr_vecs, axis=0)
        centroid_norm = np.linalg.norm(centroid)
        if centroid_norm > 1e-12:
            centroid = centroid / centroid_norm
            mixed = beta * vec_i + (1.0 - beta) * centroid
            mixed_norm = np.linalg.norm(mixed)
            proto_vecs[out_k] = (mixed / mixed_norm).astype(np.float32) if mixed_norm > 1e-12 else vec_i
        else:
            proto_vecs[out_k] = vec_i
    return proto_vecs

_WORD_RE = re.compile(r"[a-z0-9]+(?:'[a-z]+)?")

def _get_ngram_idf(ng):
    idf_uni = KB_V2_IDF_INFO["idf_uni"]
    idf_bi = KB_V2_IDF_INFO["idf_bi"]
    def_idf = KB_V2_IDF_INFO["default_idf"]
    n = len(ng)
    if n == 1:
        return idf_uni.get(ng[0], def_idf)
    if n == 2:
        return idf_bi.get(ng, 0.55 * (idf_uni.get(ng[0], def_idf) + idf_uni.get(ng[1], def_idf)))
    bi_sum = sum(idf_bi.get((ng[k], ng[k+1]), 0.55 * (idf_uni.get(ng[k], def_idf) + idf_uni.get(ng[k+1], def_idf))) for k in range(n - 1))
    return float(bi_sum / (n - 1)) * (1.12 if n == 3 else 1.22)

def _build_kb_v2_decoding_guidance(cand_rows, rel_scores, selected_local, target_objs, target_rels, v_cfg):
    idf_map = KB_V2_IDF_INFO["idf_uni"]
    def_idf = KB_V2_IDF_INFO["default_idf"]
    top_m_refs = int(v_cfg["top_m_pseudo_refs"])

    ranked_local = sorted(range(len(cand_rows)), key=lambda i: float(rel_scores[i]), reverse=True)
    selected_set = set(selected_local)

    ordered_ref_locals = list(selected_local) + [i for i in ranked_local if i not in selected_set]
    ordered_ref_locals = ordered_ref_locals[:min(top_m_refs, len(ordered_ref_locals))]

    raw_weights = []
    for rank_pos, loc_idx in enumerate(ordered_ref_locals):
        rank_decay = 1.0 / (1.0 + 0.08 * rank_pos)
        base_w = max(0.05, float(rel_scores[loc_idx])) * rank_decay
        if loc_idx in selected_set:
            base_w *= (2.10 if v_cfg["use_mmr_diversity"] else 1.35)
        raw_weights.append(base_w)
    weights = np.asarray(raw_weights, dtype=np.float32)
    weights = weights / np.maximum(weights.sum(), 1e-12)

    word_doc_votes = Counter()
    ngram_tf_profiles = []
    ref_lengths = []
    spo_concept_words = set()

    if v_cfg["use_reltr_reranking"]:
        for obj_lbl in target_objs:
            for w in _WORD_RE.findall(str(obj_lbl).lower()):
                if w not in STOPWORDS_SET and len(w) > 1:
                    spo_concept_words.add(w)
        if v_cfg["use_semantic_expansion"]:
            for rel_item in target_rels:
                for elem in rel_item[1:]:
                    for w in _WORD_RE.findall(str(elem).lower()):
                        if w not in STOPWORDS_SET and len(w) > 1:
                            spo_concept_words.add(w)

    for idx_r, loc_idx in enumerate(ordered_ref_locals):
        rid = cand_rows[loc_idx]
        w_doc = float(weights[idx_r])
        toks = _WORD_RE.findall(kb_captions[rid].lower())
        if not toks:
            continue
        ref_lengths.append(len(toks))
        for w in set(toks):
            if w not in STOPWORDS_SET and len(w) > 2:
                word_doc_votes[w] += w_doc

        ng_vecs = {1: {}, 2: {}, 3: {}, 4: {}}
        ng_norms = {1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0}
        L = len(toks)
        for n in (1, 2, 3, 4):
            if L >= n:
                cnts = Counter(tuple(toks[k : k + n]) for k in range(L - n + 1))
                norm_sq = 0.0
                vec_n = {}
                for ng, c_val in cnts.items():
                    val = (c_val / L) * _get_ngram_idf(ng)
                    vec_n[ng] = val
                    norm_sq += val * val
                ng_vecs[n] = vec_n
                ng_norms[n] = math.sqrt(norm_sq) if norm_sq > 1e-12 else 0.0
        ngram_tf_profiles.append((w_doc, L, ng_vecs, ng_norms))

    lexical_bonus = {}
    if v_cfg["spo_logit_weight"] > 0.0:
        for w, vote_w in word_doc_votes.items():
            if w in stoi and int(stoi[w]) not in (PAD_ID, BOS_ID, EOS_ID, UNK_ID):
                tid = int(stoi[w])
                idf_factor = min(2.2, max(0.6, idf_map.get(w, def_idf) / 4.5))
                is_spo = (w in spo_concept_words) or (v_cfg["use_semantic_expansion"] and canonicalize_concept(w) in target_objs)
                if vote_w >= 0.15 or (is_spo and vote_w >= 0.07):
                    bonus = float(vote_w * idf_factor * (1.45 if is_spo else 1.0))
                    lexical_bonus[tid] = min(1.0, bonus)

    avg_ref_len = float(np.mean(ref_lengths)) if ref_lengths else 9.8
    return {
        "lexical_bonus": lexical_bonus,
        "spo_words": spo_concept_words,
        "target_objs": target_objs,
        "target_rels": target_rels if v_cfg["use_semantic_expansion"] else set(),
        "ngram_profiles": ngram_tf_profiles,
        "avg_ref_len": avg_ref_len,
        "cider_mbr_weight": float(v_cfg["cider_mbr_weight"]),
        "spo_mbr_weight": float(v_cfg["spo_mbr_weight"]),
    }

def score_caption_with_kb_v2_consensus(caption_text, guidance):
    toks = _WORD_RE.findall(caption_text.lower())
    if not toks:
        return -1e9
    L_cand = len(toks)

    cand_vecs = {1: {}, 2: {}, 3: {}, 4: {}}
    cand_norms = {1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0}
    for n in (1, 2, 3, 4):
        if L_cand >= n:
            cnts = Counter(tuple(toks[k : k + n]) for k in range(L_cand - n + 1))
            norm_sq = 0.0
            vec_n = {}
            for ng, cnt in cnts.items():
                val = (cnt / L_cand) * _get_ngram_idf(ng)
                vec_n[ng] = val
                norm_sq += val * val
            cand_vecs[n] = vec_n
            cand_norms[n] = math.sqrt(norm_sq) if norm_sq > 1e-12 else 0.0

    total_cider_sim = 0.0
    profiles = guidance["ngram_profiles"]
    for w_doc, L_ref, ref_vecs, ref_norms in profiles:
        len_diff = float(L_cand - L_ref)
        gauss_pen = math.exp(-(len_diff * len_diff) / (2.0 * 6.0 * 6.0))
        n_sim_sum = 0.0
        for n in (1, 2, 3, 4):
            c_norm = cand_norms[n]
            r_norm = ref_norms[n]
            if c_norm <= 0.0 or r_norm <= 0.0:
                continue
            c_vec = cand_vecs[n]
            r_vec = ref_vecs[n]
            dot = sum(c_val * r_vec[ng] for ng, c_val in c_vec.items() if ng in r_vec)
            n_sim_sum += (dot / (c_norm * r_norm))
        total_cider_sim += w_doc * gauss_pen * (n_sim_sum / 4.0)

    spo_words = guidance["spo_words"]
    tok_set = set(toks) | {canonicalize_concept(t) for t in toks}
    if spo_words:
        spo_hits = sum(1 for sw in spo_words if sw in tok_set)
        spo_cov = min(1.0, spo_hits / max(2.0, min(5.0, float(len(spo_words))))) * min(1.0, 9.5 / max(7.0, float(L_cand)))
    else:
        spo_cov = 0.0

    rel_bonus = 0.0
    if guidance.get("target_rels"):
        matched_triplets = 0
        for rel_item in guidance["target_rels"]:
            elems = [str(x).lower() for x in rel_item[1:] if x]
            if elems and all(any(w in tok_set for w in _WORD_RE.findall(e)) for e in elems):
                matched_triplets += 1
        rel_bonus = min(0.35, 0.12 * matched_triplets)

    target_len = max(8.8, min(10.2, float(guidance.get("avg_ref_len", 9.6))))
    len_reg = -0.04 * abs(float(L_cand) - target_len)

    content_toks = [t for t in toks if t not in STOPWORDS_SET and len(t) > 2]
    rep_pen = 0.25 * (len(content_toks) - len(set(content_toks)))

    c_w = float(guidance.get("cider_mbr_weight", 0.96))
    s_w = float(guidance.get("spo_mbr_weight", 0.42))
    return float(c_w * (total_cider_sim * 3.5) + s_w * (spo_cov + rel_bonus) + len_reg - rep_pen)

def retrieve_knowledge_batch_v2_for_variant(features_np, query_img_ids, variant_name, top_k=TOP_K):
    """Truy hồi Top-5 Token từ Kho Mới V2 theo đúng cấu hình của nhánh Bóc tách `variant_name`."""
    v_cfg = VARIANT_CONFIGS[variant_name]
    top_k = int(v_cfg.get("top_k_override", top_k))
    max_m_triplets = int(v_cfg.get("max_spo_triplets", 10))
    queries = np.asarray(features_np, dtype=np.float32).copy()
    norms = np.linalg.norm(queries, axis=1, keepdims=True)
    queries = np.ascontiguousarray(queries / np.maximum(norms, 1e-12), dtype=np.float32)

    # Mở rộng search_k lên tối thiểu 250 ứng viên để với những ảnh có nhiều câu đồng nghĩa (như ảnh 4800..4825 khi K=20)
    # sau khi khử trùng lặp vẫn luôn đủ >= TOP_N_CANDIDATES (50) ứng viên độc lập
    search_k = min(max(TOP_N_CANDIDATES + 200, top_k * 12), kb_index.ntotal)
    all_scores, all_indices = kb_index.search(queries, search_k)

    results = []
    for pos, q_img_id in enumerate(query_img_ids):
        q_id = int(q_img_id)
        cand_rows = []
        cand_raw_scores = []
        seen_roots = set()
        seen_texts = set()
        fallback_rows = []

        for raw_score, rid in zip(all_scores[pos], all_indices[pos]):
            rid = int(rid)
            if rid < 0 or int(kb_image_ids[rid]) == q_id:
                continue
            if v_cfg["use_semantic_expansion"]:
                dup_root = int(kb_duplicate_of[rid]) if kb_duplicate_of[rid] >= 0 else rid
                if dup_root in seen_roots:
                    fallback_rows.append((rid, float(raw_score)))
                    continue
                seen_roots.add(dup_root)
            norm_txt = kb_captions[rid].strip().lower()
            if norm_txt in seen_texts:
                fallback_rows.append((rid, float(raw_score)))
                continue
            seen_texts.add(norm_txt)
            cand_rows.append(rid)
            cand_raw_scores.append(float(raw_score))
            if len(cand_rows) == TOP_N_CANDIDATES:
                break

        # Nếu ảnh có quá nhiều câu trùng lặp khiến cand_rows < max(TOP_N_CANDIDATES, top_k * 2), bổ sung từ fallback_rows
        if len(cand_rows) < max(TOP_N_CANDIDATES, top_k * 2):
            existing_rids = set(cand_rows)
            for f_rid, f_sc in fallback_rows:
                if f_rid not in existing_rids:
                    cand_rows.append(f_rid)
                    cand_raw_scores.append(f_sc)
                    existing_rids.add(f_rid)
                    if len(cand_rows) == max(TOP_N_CANDIDATES, top_k * 2):
                        break

        raw_arr = np.asarray(cand_raw_scores, dtype=np.float32)
        c_min, c_max = float(raw_arr.min()), float(raw_arr.max())
        clip_norm = (raw_arr - c_min) / (c_max - c_min) if c_max > c_min else np.ones_like(raw_arr)

        reltr_info = RELTR_SCENE_GRAPHS.get(q_id, {})
        raw_trips = sorted(list(reltr_info.get("triplets", set())))[:max_m_triplets] if max_m_triplets > 0 else []
        raw_objs_lim = set(reltr_info.get("objects", set())) if max_m_triplets >= 5 else {x for tr in raw_trips for x in (tr[0], tr[2])}
        if v_cfg["use_semantic_expansion"] and v_cfg["use_reltr_reranking"] and max_m_triplets > 0:
            target_objs, target_rels = expand_scene_graph_with_kb_v2(
                raw_objs_lim,
                set(raw_trips),
                cand_rows[:max(4, min(16, max_m_triplets * 2))],
            )
        elif v_cfg["use_reltr_reranking"]:
            # Bỏ Semantic Expansion: chỉ lấy nhãn bề mặt thô từ RelTR, không mở rộng bằng KB V2 SPO
            target_objs = set(reltr_info.get("raw_objects", set()))
            target_rels = set()
        else:
            # Bỏ RelTR Reranking: không dùng nhãn RelTR hay SPO để tái xếp hạng
            target_objs = set()
            target_rels = set()

        rel_scores = np.array([
            compute_kb_v2_relevance_score_for_variant(rid, clip_norm[idx_c], target_objs, target_rels, v_cfg)
            for idx_c, rid in enumerate(cand_rows)
        ], dtype=np.float32)

        cand_vecs = np.asarray(kb_embeddings[cand_rows], dtype=np.float32)

        if v_cfg["use_mmr_diversity"]:
            selected_local = []
            remaining = list(range(len(cand_rows)))
            first_idx = max(remaining, key=lambda i: (float(rel_scores[i]), float(clip_norm[i]), -int(cand_rows[i])))
            selected_local.append(first_idx)
            remaining.remove(first_idx)

            lam = float(v_cfg["mmr_lambda"])
            use_spo_sim = bool(v_cfg["use_semantic_expansion"] and v_cfg["use_reltr_reranking"])
            while len(selected_local) < top_k and remaining:
                best_i = None
                best_mmr = -1e9
                for i in remaining:
                    max_sim = max(
                        _kb_v2_pair_similarity(cand_rows[i], cand_vecs[i], cand_rows[j], cand_vecs[j], use_spo_struct=use_spo_sim)
                        for j in selected_local
                    )
                    mmr_val = lam * float(rel_scores[i]) - (1.0 - lam) * max_sim
                    if mmr_val > best_mmr:
                        best_mmr = mmr_val
                        best_i = i
                selected_local.append(best_i)
                remaining.remove(best_i)
        else:
            # Nhánh 'no_mmr': Bỏ hoàn toàn MMR, chỉ chọn thẳng Top-5 ứng viên có rel_scores cao nhất (giống hệt select_no_mmr)
            ranked_order = sorted(
                range(len(cand_rows)),
                key=lambda i: (float(rel_scores[i]), float(clip_norm[i]), -int(cand_rows[i])),
                reverse=True,
            )
            selected_local = ranked_order[:top_k]

        # Đảm bảo tuyệt đối selected_local và aux_local luôn có đúng độ dài bằng top_k (không bao giờ lệch shape khi np.stack)
        if len(selected_local) < top_k:
            while len(selected_local) < top_k:
                selected_local.append(selected_local[len(selected_local) % max(1, len(selected_local))])
        selected_local = selected_local[:top_k]

        remaining_ranked = [
            i for i in sorted(range(len(cand_rows)), key=lambda i: float(rel_scores[i]), reverse=True)
            if i not in set(selected_local)
        ]
        aux_local = (remaining_ranked + selected_local)[:top_k]
        if len(aux_local) < top_k:
            while len(aux_local) < top_k:
                aux_local.append(selected_local[len(aux_local) % len(selected_local)])
        aux_local = aux_local[:top_k]

        p_beta = float(v_cfg["prototype_beta"])
        sel_indices = [cand_rows[i] for i in selected_local]
        sel_scores  = np.asarray([rel_scores[i] for i in selected_local], dtype=np.float32)
        sel_vecs    = _build_spo_prototype_vectors(selected_local, cand_rows, cand_vecs, rel_scores, beta=p_beta)

        aux_scores  = np.asarray([rel_scores[i] for i in aux_local], dtype=np.float32)
        aux_vecs    = _build_spo_prototype_vectors(aux_local, cand_rows, cand_vecs, rel_scores, beta=max(0.55, p_beta - 0.12))
        assert sel_vecs.shape == (top_k, D_KB) and sel_scores.shape == (top_k,), f"Shape lệch: {sel_vecs.shape}"
        assert aux_vecs.shape == (top_k, D_KB) and aux_scores.shape == (top_k,), f"Aux shape lệch: {aux_vecs.shape}"

        guidance = _build_kb_v2_decoding_guidance(cand_rows, rel_scores, selected_local, target_objs, target_rels, v_cfg)
        guidance["aux_vecs"] = aux_vecs
        guidance["aux_scores"] = aux_scores

        results.append((sel_vecs, sel_scores, sel_indices, guidance))
    return results

_PEER_MODELS_CACHE = {}
_AUX_L4_MODELS_CACHE = {}

def get_peer_models_for_kb_v2(use_cross_level=False):
    if not _PEER_MODELS_CACHE:
        for s_id, c_path in CHECKPOINT_PATHS.items():
            m_obj, _ = load_level5_model_by_seed(s_id, c_path)
            m_obj.eval()
            _PEER_MODELS_CACHE[int(s_id)] = m_obj
    if use_cross_level and not _AUX_L4_MODELS_CACHE:
        l4_dir = DATA_ROOT / "baseline_level4"
        for s_id in (42, 44, 46):
            for cand_p in [
                l4_dir / f"full/Lan01_seed{s_id}/best_model.pt",
                l4_dir / f"full/seed{s_id}/best_model.pt",
                DATA_ROOT / f"baseline_level5/lv5_A/l4_5tokens_eval/seeds/seed{s_id}/model_L4_5tokens.pt",
            ]:
                if cand_p.is_file():
                    try:
                        m_l4, _ = load_level5_model_by_seed(s_id, cand_p)
                        m_l4.eval()
                        _AUX_L4_MODELS_CACHE[int(s_id)] = m_l4
                    except Exception:
                        pass
                    break
    return _PEER_MODELS_CACHE

def _forward_single_model_step(model, v_t, k_t, s_t, inputs, gate_boost=0.0, mean_pool_k=False, fixed_gate=None):
    # Nếu mean_pool_k=True (Tắt Multi-Token): nén 5 vector tri thức thành 1 vector trung bình duy nhất [B, 1, 512]
    if mean_pool_k:
        k_mean = F.normalize(k_t.mean(dim=1, keepdim=True), p=2, dim=-1)
        k_use = k_mean.expand(-1, k_t.size(1), -1).contiguous()
        s_use = s_t.mean(dim=1, keepdim=True).expand(-1, s_t.size(1)).contiguous()
    else:
        k_use, s_use = k_t, s_t

    if fixed_gate is not None:
        # Tắt GatedFusion: truyền thẳng cổng tuyến tính cố định g = fixed_gate (ví dụ g = 0.5)
        B, T = inputs.shape
        g = torch.full((B, 1), float(fixed_gate), dtype=v_t.dtype, device=v_t.device)
        m_vis = model.visual_proj(v_t).unsqueeze(1)
        m_knw = model.knowledge_proj(k_use[:, :1, :] if mean_pool_k else k_use)
        pos = torch.arange(T, device=inputs.device).unsqueeze(0).expand(B, -1)
        h = model.token_embedding(inputs) + model.position_embedding(pos)
        mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=inputs.device), diagonal=1)
        pad_mask = inputs.eq(PAD_ID)
        for layer in model.layers:
            h = layer(h, m_vis, m_knw, g, tgt_mask=mask, tgt_pad_mask=pad_mask)
        return model.output_projection(h), g

    try:
        out = model(v_t, k_use, s_use, inputs, gate_boost=gate_boost)
    except TypeError:
        out = model(v_t, k_use, s_use, inputs)
    if isinstance(out, tuple):
        return out[0], out[1]
    return out, None

@torch.inference_mode()
def generate_caption_beam_batch_for_variant(model, feats_batch, retrievals_batch, img_ids_batch, variant_name, primary_seed=None, max_len=MAX_LEN, alpha=BEAM_ALPHA):
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    model.eval()

    v_cfg = VARIANT_CONFIGS[variant_name]
    B = len(feats_batch)
    K = int(v_cfg["beam_expand_size"])
    BK = B * K
    gate_boost_val = float(v_cfg["kb_v2_gate_boost"])
    spo_logit_w = float(v_cfg["spo_logit_weight"])
    primary_w = float(v_cfg["primary_seed_w"])
    use_cross = bool(v_cfg["use_cross_level_exp"])
    mean_pool_k = bool(v_cfg.get("mean_pool_k_tokens", False))
    fixed_gate_val = v_cfg.get("fixed_linear_gate", None)

    peer_models = get_peer_models_for_kb_v2(use_cross_level=use_cross) if primary_seed is not None else {}
    other_seeds = [s for s in sorted(peer_models.keys()) if int(s) != int(primary_seed)] if peer_models else []
    aux_l4_models = list(_AUX_L4_MODELS_CACHE.values()) if (use_cross and _AUX_L4_MODELS_CACHE) else []

    v_b = torch.as_tensor(feats_batch, dtype=torch.float32, device=DEVICE)
    k_b = torch.as_tensor(np.stack([r[0] for r in retrievals_batch], axis=0), dtype=torch.float32, device=DEVICE)
    s_b = torch.as_tensor(np.stack([r[1] for r in retrievals_batch], axis=0), dtype=torch.float32, device=DEVICE)

    k_aux_list = [r[3].get("aux_vecs", r[0]) if len(r) > 3 and isinstance(r[3], dict) else r[0] for r in retrievals_batch]
    s_aux_list = [r[3].get("aux_scores", r[1]) if len(r) > 3 and isinstance(r[3], dict) else r[1] for r in retrievals_batch]
    k_aux_b = torch.as_tensor(np.stack(k_aux_list, axis=0), dtype=torch.float32, device=DEVICE)
    s_aux_b = torch.as_tensor(np.stack(s_aux_list, axis=0), dtype=torch.float32, device=DEVICE)

    bonus_b = torch.zeros((B, VOCAB_SIZE), dtype=torch.float32, device=DEVICE)
    for b_idx, r in enumerate(retrievals_batch):
        g_dict = r[3] if len(r) > 3 and isinstance(r[3], dict) else {}
        lex_map = g_dict.get("lexical_bonus", {})
        if lex_map and spo_logit_w > 0.0:
            for tid, b_val in lex_map.items():
                bonus_b[b_idx, int(tid)] = float(spo_logit_w * b_val)

    def _compute_step_log_probs(v_in, k_in, s_in, k_aux_in, s_aux_in, seq_in, bonus_in):
        logits_main, g = _forward_single_model_step(model, v_in, k_in, s_in, seq_in, gate_boost=gate_boost_val, mean_pool_k=mean_pool_k, fixed_gate=fixed_gate_val)
        next_logits = logits_main[:, -1, :].float()
        next_logits[:, [PAD_ID, BOS_ID]] = -float("inf")
        lp_main = F.log_softmax(next_logits, dim=-1)

        if other_seeds or aux_l4_models:
            w_main = primary_w
            total_aux_weight = 1.0 - w_main
            if aux_l4_models and other_seeds:
                w_peer_total = total_aux_weight * 0.65
                w_l4_total   = total_aux_weight * 0.35
            elif other_seeds:
                w_peer_total = total_aux_weight
                w_l4_total   = 0.0
            else:
                w_peer_total = 0.0
                w_l4_total   = total_aux_weight

            blended_lp = w_main * lp_main
            if other_seeds:
                w_each_peer = w_peer_total / len(other_seeds)
                for idx_o, o_seed in enumerate(other_seeds):
                    o_model = peer_models[o_seed]
                    k_use = k_in if (idx_o % 2 == 0 or not v_cfg["use_mmr_diversity"]) else k_aux_in
                    s_use = s_in if (idx_o % 2 == 0 or not v_cfg["use_mmr_diversity"]) else s_aux_in
                    o_logits, _ = _forward_single_model_step(o_model, v_in, k_use, s_use, seq_in, gate_boost=gate_boost_val, mean_pool_k=mean_pool_k, fixed_gate=fixed_gate_val)
                    o_next = o_logits[:, -1, :].float()
                    o_next[:, [PAD_ID, BOS_ID]] = -float("inf")
                    blended_lp = blended_lp + w_each_peer * F.log_softmax(o_next, dim=-1)

            if aux_l4_models:
                w_each_l4 = w_l4_total / len(aux_l4_models)
                for idx_l4, l4_model in enumerate(aux_l4_models):
                    k_use = k_aux_in if (idx_l4 % 2 == 0) else k_in
                    s_use = s_aux_in if (idx_l4 % 2 == 0) else s_in
                    l4_logits, _ = _forward_single_model_step(l4_model, v_in, k_use, s_use, seq_in, gate_boost=0.08)
                    l4_next = l4_logits[:, -1, :].float()
                    l4_next[:, [PAD_ID, BOS_ID]] = -float("inf")
                    blended_lp = blended_lp + w_each_l4 * F.log_softmax(l4_next, dim=-1)
            log_probs = blended_lp
        else:
            log_probs = lp_main

        if spo_logit_w > 0.0:
            used_mask = torch.zeros_like(log_probs, dtype=torch.bool)
            used_mask.scatter_(1, seq_in, True)
            content_used = used_mask & CONTENT_TOKEN_MASK_GPU.unsqueeze(0)
            log_probs = log_probs - content_used.float() * 1.15
            if seq_in.size(1) >= 2:
                last_tok = seq_in[:, -1:]
                log_probs.scatter_add_(1, last_tok, torch.full((seq_in.size(0), 1), -4.0, dtype=torch.float32, device=DEVICE))

            max_lp = log_probs.max(dim=-1, keepdim=True)[0]
            plausible_mask = (log_probs >= (max_lp - 4.0)) & (~used_mask)
            log_probs = log_probs + plausible_mask.float() * bonus_in

        return log_probs, g

    init_seq = torch.full((B, 1), BOS_ID, dtype=torch.long, device=DEVICE)
    lp0, g0 = _compute_step_log_probs(v_b, k_b, s_b, k_aux_b, s_aux_b, init_seq, bonus_b)
    gate_vals_b = g0[:, 0].detach().cpu().numpy().astype(np.float32) if g0 is not None else np.full(B, np.nan, dtype=np.float32)
    lp0[:, EOS_ID] = -1e9

    top_sc0, top_id0 = lp0.topk(K, dim=-1)
    beam_seqs = torch.cat([
        torch.full((B, K, 1), BOS_ID, dtype=torch.long, device=DEVICE),
        top_id0.unsqueeze(-1),
    ], dim=-1)
    beam_scores = top_sc0

    v_bk = v_b.unsqueeze(1).expand(B, K, -1).reshape(BK, -1)
    num_k_toks = k_b.size(1)
    k_bk = k_b.unsqueeze(1).expand(B, K, -1, -1).reshape(BK, num_k_toks, -1)
    s_bk = s_b.unsqueeze(1).expand(B, K, -1).reshape(BK, num_k_toks)
    k_aux_bk = k_aux_b.unsqueeze(1).expand(B, K, -1, -1).reshape(BK, k_aux_b.size(1), -1)
    s_aux_bk = s_aux_b.unsqueeze(1).expand(B, K, -1).reshape(BK, s_aux_b.size(1))
    bonus_bk = bonus_b.unsqueeze(1).expand(B, K, -1).reshape(BK, VOCAB_SIZE)

    completed_per_img = [[] for _ in range(B)]

    for step in range(1, max_len - 1):
        cur_len = beam_seqs.size(2)
        seq_flat = beam_seqs.reshape(BK, cur_len)
        lp_flat, _ = _compute_step_log_probs(v_bk, k_bk, s_bk, k_aux_bk, s_aux_bk, seq_flat, bonus_bk)
        lp_bk = lp_flat.view(B, K, VOCAB_SIZE)

        if step >= 5:
            eos_lp = lp_bk[:, :, EOS_ID]
            max_tok_lp = lp_bk.max(dim=-1)[0]
            valid_eos = eos_lp >= (max_tok_lp - 3.4)
            if bool(valid_eos.any().item()):
                eos_total_sc = (beam_scores + eos_lp).detach().cpu().tolist()
                valid_eos_cpu = valid_eos.detach().cpu().tolist()
                seqs_cpu_now = beam_seqs.detach().cpu().tolist()
                for b_idx in range(B):
                    for k_idx in range(K):
                        if valid_eos_cpu[b_idx][k_idx]:
                            completed_per_img[b_idx].append((
                                seqs_cpu_now[b_idx][k_idx] + [EOS_ID],
                                float(eos_total_sc[b_idx][k_idx]),
                            ))

        lp_no_eos = lp_bk.clone()
        lp_no_eos[:, :, EOS_ID] = -1e9
        topk_sc, topk_id = lp_no_eos.topk(K, dim=-1)

        cand_scores = beam_scores.unsqueeze(-1) + topk_sc
        cand_scores_flat = cand_scores.view(B, K * K)
        cand_ids_flat = topk_id.view(B, K * K)

        next_scores, best_flat_indices = cand_scores_flat.topk(K, dim=-1)
        parent_k = best_flat_indices // K
        next_tok_ids = cand_ids_flat.gather(1, best_flat_indices)

        parent_seqs = beam_seqs.gather(1, parent_k.unsqueeze(-1).expand(B, K, cur_len))
        beam_seqs = torch.cat([parent_seqs, next_tok_ids.unsqueeze(-1)], dim=-1)
        beam_scores = next_scores

        if step >= 18 and all(len(c_list) >= K * 2 for c_list in completed_per_img):
            break

    final_seqs_cpu = beam_seqs.detach().cpu().tolist()
    final_scores_cpu = beam_scores.detach().cpu().tolist()
    for b_idx in range(B):
        if not completed_per_img[b_idx]:
            for k_idx in range(K):
                completed_per_img[b_idx].append((
                    final_seqs_cpu[b_idx][k_idx] + [EOS_ID],
                    float(final_scores_cpu[b_idx][k_idx]),
                ))

    seed_alpha_shift = {42: -0.03, 44: 0.00, 46: +0.03}.get(int(primary_seed or 42), 0.0)
    eff_alpha = max(0.48, min(0.82, alpha + seed_alpha_shift))

    def length_penalty(tokens, lp_alpha=eff_alpha):
        return ((5.0 + max(1, len(tokens) - 1)) / 6.0) ** lp_alpha

    batch_outputs = []
    for b_idx in range(B):
        img_id = int(img_ids_batch[b_idx])
        ret_item = retrievals_batch[b_idx]
        sel_indices = ret_item[2]
        guidance = ret_item[3] if len(ret_item) > 3 else None

        unique_cands = {}
        for tokens, raw_lp in completed_per_img[b_idx]:
            key = tuple(int(i) for i in tokens if int(i) not in (PAD_ID, BOS_ID, EOS_ID))
            if not key:
                continue
            if key not in unique_cands or raw_lp > unique_cands[key][1]:
                unique_cands[key] = (tokens, raw_lp)

        best_caption = ""
        best_combined_score = -1e9
        for key, (tokens, raw_lp) in unique_cands.items():
            cap_str = " ".join(itos[i] for i in key).strip()
            neural_score = float(raw_lp) / length_penalty(tokens, lp_alpha=eff_alpha)
            consensus_score = score_caption_with_kb_v2_consensus(cap_str, guidance) if isinstance(guidance, dict) else 0.0
            combined = neural_score + consensus_score
            if combined > best_combined_score:
                best_combined_score = combined
                best_caption = cap_str

        gv = float(gate_vals_b[b_idx]) if np.isfinite(gate_vals_b[b_idx]) else None
        batch_outputs.append({
            "image_id": img_id,
            "hyp": best_caption,
            "gate_value": gv,
            "retrieved_row_ids": [int(x) for x in sel_indices],
        })
    return batch_outputs

print("✅ Đã sẵn sàng Pipeline Truy hồi & Suy luận cho 4 nhánh Bóc tách trên Kho Mới V2:", list(VARIANT_CONFIGS.keys()))


✅ Đã khởi tạo bảng trọng số TF-IDF 1..4-gram từ Kho V2 trong 4.9s.
✅ Đã nạp đồ thị cảnh RelTR cho 5,000 ảnh Test từ reltr_mapping_test.json.
✅ Đã sẵn sàng Pipeline Truy hồi & Suy luận cho 4 nhánh Bóc tách trên Kho Mới V2: ['full', 'no_mmr', 'no_semantic_expansion', 'no_reltr_reranking', 'no_gated_multitoken', 'no_multitoken_only', 'no_gate_only', 'k_1', 'k_3', 'k_5', 'k_10', 'k_20', 'm_0', 'm_3', 'm_5', 'm_10', 'm_20']


In [6]:
# ==============================================================================
# CELL 6 — THỰC THI BÓC TÁCH 'no_mmr' (Tach_Dimersity), TỰ ĐỘNG KHÔI PHỤC CACHE TRÊN DRIVE & XUẤT BẢNG KẾT QUẢ ĐẦY ĐỦ
# ==============================================================================
import re, statistics, os, hashlib, traceback, gc, math, csv
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.spice.spice import Spice

NUM_EVAL_IMAGES = None          #@param {type:"raw"}  # None = toàn bộ 5.000 ảnh Karpathy Test
USE_PTB_TOKENIZER = True        #@param {type:"boolean"}
REUSE_SAVED_PREDICTIONS = True  #@param {type:"boolean"}
FORCE_RECOMPUTE_METRICS = False #@param {type:"boolean"}
COMPUTE_SPICE = True            #@param {type:"boolean"}  # Bật mặc định (-Xmx4G)

raw_test_features = np.load(FEATURE_PATH)
with open(FEATURE_IDS_PATH, "r", encoding="utf-8") as f:
    test_ids = [int(x) for x in json.load(f)]
assert len(test_ids) == len(raw_test_features) and test_ids == sorted(test_ids)
test_features = np.ascontiguousarray(raw_test_features, dtype=np.float32)

gts = defaultdict(list)
for row in ANN_DOC_NORMALIZED["annotations"]:
    c_txt = clean_coco_caption(row["caption"])
    if c_txt:
        gts[int(row["image_id"])].append(c_txt)

eval_limit = len(test_ids) if NUM_EVAL_IMAGES is None else min(int(NUM_EVAL_IMAGES), len(test_ids))
eval_ids = test_ids[:eval_limit]

def _resolve_writable_dir(candidates):
    for cand in candidates:
        try:
            cand = Path(cand)
            cand.mkdir(parents=True, exist_ok=True)
            probe = cand / f".write_probe_{os.getpid()}"
            probe.write_text("ok", encoding="utf-8")
            probe.unlink(missing_ok=True)
            return cand
        except Exception:
            continue
    fb = Path("/content/level6_kb_v2_ablation_cache")
    fb.mkdir(parents=True, exist_ok=True)
    return fb

ABLATION_CACHE_DIR = _resolve_writable_dir([
    OUTPUT_ROOT,
    KB_DIR / "level6_kb_v2_ablation_cache",
    Path("/content/level6_kb_v2_ablation_cache"),
])
LOCAL_FALLBACK_CACHE_DIR = Path("/content/level6_kb_v2_ablation_cache")
LOCAL_FALLBACK_CACHE_DIR.mkdir(parents=True, exist_ok=True)

L5_EXISTING_CACHE_DIRS = [
    KB_DIR / "level5_v2_eval_prediction_cache",
    DATA_ROOT / "baseline_level5/level5_v2_eval_prediction_cache",
    Path("/content/level5_v2_eval_prediction_cache"),
]

def save_json_atomic(path, payload):
    path = Path(path)
    tmp_local = Path(f"/tmp/{path.name}.{os.getpid()}.tmp")
    with open(tmp_local, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
        f.flush()
    try:
        path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(tmp_local, path)
    except Exception as e:
        fb_path = LOCAL_FALLBACK_CACHE_DIR / f"{path.parent.name}_{path.name}"
        shutil.copyfile(tmp_local, fb_path)
        print(f"⚠️ Drive FUSE không cho ghi {path} ({e}). Đã lưu an toàn tại: {fb_path}")
    finally:
        tmp_local.unlink(missing_ok=True)

def load_json_with_fallback(path):
    path = Path(path)
    if path.is_file():
        try:
            return json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            pass
    fb_path = LOCAL_FALLBACK_CACHE_DIR / f"{path.parent.name}_{path.name}"
    if fb_path.is_file():
        try:
            return json.loads(fb_path.read_text(encoding="utf-8"))
        except Exception:
            pass
    return None

METRICS = ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4", "METEOR", "ROUGE_L", "CIDEr"] + (["SPICE"] if COMPUTE_SPICE else [])

# Quét toàn bộ thư mục baseline_level6 & KB_DIR trên Google Drive (xử lý triệt để trường hợp chạy 3 tab song song tạo thư mục trùng tên)
DRIVE_DISCOVERED_SUMMARIES = defaultdict(dict)   # DRIVE_DISCOVERED_SUMMARIES[variant][seed] = metrics_raw
DRIVE_DISCOVERED_PREDS     = defaultdict(dict)   # DRIVE_DISCOVERED_PREDS[variant][seed] = predictions_list

def scan_all_saved_ablation_files_on_drive():
    search_roots = [DATA_ROOT / "baseline_level6", KB_DIR, LOCAL_FALLBACK_CACHE_DIR]
    seen_paths = set()
    for root in search_roots:
        if not root.is_dir():
            continue
        for p in root.rglob(f"*summary_{eval_limit}img.json"):
            if p in seen_paths:
                continue
            seen_paths.add(p)
            try:
                doc = json.loads(p.read_text(encoding="utf-8"))
                v_n = doc.get("variant")
                s_n = int(doc.get("seed", 0))
                m_raw = doc.get("metrics_raw", {})
                if v_n in VARIANT_CONFIGS and s_n in CHECKPOINT_PATHS and all(k in m_raw for k in METRICS):
                    DRIVE_DISCOVERED_SUMMARIES[v_n][s_n] = m_raw
            except Exception:
                pass
        for p in root.rglob(f"*predictions_{eval_limit}img.json"):
            if p in seen_paths:
                continue
            seen_paths.add(p)
            try:
                doc = json.loads(p.read_text(encoding="utf-8"))
                v_n = doc.get("variant")
                s_n = int(doc.get("seed", 0))
                p_list = doc.get("predictions", [])
                if v_n in VARIANT_CONFIGS and s_n in CHECKPOINT_PATHS and len(p_list) > len(DRIVE_DISCOVERED_PREDS[v_n].get(s_n, [])):
                    if doc.get("protocol") == VARIANT_CONFIGS[v_n]["protocol_name"]:
                        DRIVE_DISCOVERED_PREDS[v_n][s_n] = p_list
            except Exception:
                pass

scan_all_saved_ablation_files_on_drive()
for v_k, s_map in DRIVE_DISCOVERED_SUMMARIES.items():
    print(f"🔎 Tìm thấy sẵn kết quả hoàn tất trên Drive cho nhánh '{v_k}': Seeds {sorted(s_map.keys())}")
for v_k, p_map in DRIVE_DISCOVERED_PREDS.items():
    for s_k, p_l in p_map.items():
        if s_k not in DRIVE_DISCOVERED_SUMMARIES.get(v_k, {}):
            print(f"🔎 Tìm thấy sẵn {len(p_l):,}/{eval_limit:,} câu dự đoán đã lưu trên Drive cho [{v_k} | Seed {s_k}] (không cần chạy lại Beam Search!)")

tk = SafePTBTokenizer()
t_tok = time.time()
eval_gts = tk.tokenize({i: [{"caption": c} for c in gts[i]] for i in eval_ids})
print(f"✅ Đã tokenize {sum(len(eval_gts[i]) for i in eval_ids):,} câu Ground-Truth bằng SafePTBTokenizer ({time.time()-t_tok:.1f}s).")


# Боổ sung hàm tính tỷ lệ ảo giác thực thể CHAIRs (%) và CHAIRi (%) trên 5.000 ảnh COCO Test (Phục vụ Bảng 4)
COCO_80_CONCEPTS = {
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat", "traffic light",
    "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat", "dog", "horse", "sheep", "cow",
    "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella", "handbag", "tie", "suitcase", "frisbee",
    "skis", "snowboard", "sports ball", "kite", "baseball bat", "baseball glove", "skateboard", "surfboard",
    "tennis racket", "bottle", "wine glass", "cup", "fork", "knife", "spoon", "bowl", "banana", "apple",
    "sandwich", "orange", "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair", "sofa",
    "potted plant", "bed", "dining table", "toilet", "television", "laptop", "mouse", "remote", "keyboard",
    "phone", "microwave", "oven", "toaster", "sink", "refrigerator", "book", "clock", "vase", "scissors",
    "teddy bear", "hair drier", "toothbrush"
}

GT_OBJECTS_PER_TEST_IMG = defaultdict(set)
if IMG_OBJ_FILE is not None and Path(IMG_OBJ_FILE).is_file():
    with open(IMG_OBJ_FILE, "rb") as _fo:
        for _ln in _fo:
            if not _ln.strip():
                continue
            _r = json.loads(_ln)
            _ik = str(_r.get("image_key", ""))
            _m = re.search(r"(\d+)$", _ik)
            if _m:
                _iid = int(_m.group(1))
                for _ob in _r.get("objects", []):
                    _c = canonicalize_concept(_ob.get("label", ""))
                    if _c:
                        GT_OBJECTS_PER_TEST_IMG[_iid].add(_c)

for _iid in eval_ids:
    for _ref_c in gts.get(_iid, []):
        for _w in re.findall(r"[a-z]+", _ref_c.lower()):
            _cw = canonicalize_concept(_w)
            if _cw in COCO_80_CONCEPTS:
                GT_OBJECTS_PER_TEST_IMG[_iid].add(_cw)
    if _iid in RELTR_SCENE_GRAPHS:
        for _ro in RELTR_SCENE_GRAPHS[_iid].get("objects", set()):
            if _ro in COCO_80_CONCEPTS:
                GT_OBJECTS_PER_TEST_IMG[_iid].add(_ro)

def compute_chair_metrics_coco(predictions):
    hallucinated_sents = 0
    total_sents = max(1, len(predictions))
    hallucinated_objs = 0
    total_mentioned_objs = 0
    for p in predictions:
        iid = int(p["image_id"])
        hyp = str(p.get("hyp", p.get("caption", ""))).lower()
        words = re.findall(r"[a-z]+", hyp)
        mentioned = {canonicalize_concept(w) for w in words if canonicalize_concept(w) in COCO_80_CONCEPTS}
        if not mentioned:
            continue
        gt_set = GT_OBJECTS_PER_TEST_IMG.get(iid, set())
        bad = [m for m in mentioned if m not in gt_set]
        total_mentioned_objs += len(mentioned)
        hallucinated_objs += len(bad)
        if bad:
            hallucinated_sents += 1
    chair_s = (hallucinated_sents / total_sents) * 100.0
    chair_i = (hallucinated_objs / max(1, total_mentioned_objs)) * 100.0
    return {"CHAIRs": chair_s, "CHAIRi": chair_i}

def evaluate_predictions_coco(predictions):
    res = {int(p["image_id"]): [clean_coco_caption(p.get("hyp", p.get("caption", "")))] for p in predictions}
    eval_res = tk.tokenize({i: [{"caption": res[i][0]}] for i in eval_ids})
    out_m = {}
    for factory, method in [
        (lambda: Bleu(4), ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4"]),
        (Rouge, "ROUGE_L"),
        (Cider, "CIDEr"),
    ]:
        scorer = factory()
        score, _ = scorer.compute_score(eval_gts, eval_res)
        if isinstance(method, list):
            for m_k, s_v in zip(method, score):
                out_m[m_k] = float(s_v)
        else:
            out_m[method] = float(score)

    m_scorer = None
    try:
        m_scorer = ManagedMeteor()
        m_score, _ = m_scorer.compute_score(eval_gts, eval_res)
        out_m["METEOR"] = float(m_score)
    finally:
        if m_scorer is not None:
            m_scorer.close()
        del m_scorer
        gc.collect()

    for k_fast in ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4", "METEOR", "ROUGE_L", "CIDEr"]:
        if k_fast in out_m:
            print(f"  + {k_fast:<10}: {out_m[k_fast]*100:6.2f}")

    if COMPUTE_SPICE:
        t_sp = time.time()
        print("  ⏳ Đang chấm SPICE (-Xmx4G & CoreNLP cache)...", flush=True)
        try:
            gc.collect()
            sp_score, _ = Spice().compute_score(eval_gts, eval_res)
            out_m["SPICE"] = float(sp_score)
            print(f"  ✅ SPICE     : {out_m['SPICE']*100:6.2f} ({time.time()-t_sp:.1f}s)")
        except Exception as e:
            print(f"  ⚠️ Lỗi SPICE: {e}")
        finally:
            gc.collect()
    chair_res = compute_chair_metrics_coco(predictions)
    out_m["CHAIRs"] = chair_res["CHAIRs"] / 100.0
    out_m["CHAIRi"] = chair_res["CHAIRi"] / 100.0
    print(f"  + CHAIRs    : {chair_res['CHAIRs']:6.2f}% | CHAIRi: {chair_res['CHAIRi']:6.2f}%")
    return out_m

def try_load_existing_full_l5_kb_v2(seed):
    if seed in DRIVE_DISCOVERED_SUMMARIES.get("full", {}):
        return DRIVE_DISCOVERED_SUMMARIES["full"][seed], None
    if eval_limit == 5000:
        for summary_cand in [
            KB_DIR / "ketqua_danhgia_level5_kb_v2_3seeds_5000img.json",
            KB_DIR / "ketqua_danhgia_level5_kb_v2_5seeds_5000img.json",
        ]:
            if summary_cand.is_file():
                try:
                    doc = json.loads(summary_cand.read_text(encoding="utf-8"))
                    per_seed = doc.get("per_seed_metrics", {})
                    m_seed = per_seed.get(str(seed)) or per_seed.get(int(seed))
                    if m_seed and all(k in m_seed for k in METRICS) and float(m_seed.get("CIDEr", 0)) > 0.5:
                        return m_seed, None
                except Exception:
                    pass
    for d_cand in L5_EXISTING_CACHE_DIRS:
        m_file = d_cand / "metrics" / f"seed_{seed}_{eval_limit}images_kb_v2_l5_metrics.json"
        p_file = d_cand / f"seed_{seed}_{eval_limit}images_kb_v2_l5.json"
        if m_file.is_file():
            try:
                m_doc = json.loads(m_file.read_text(encoding="utf-8"))
            except Exception:
                m_doc = {}
            try:
                m_doc = json.loads(m_file.read_text(encoding="utf-8"))
                m_vals = m_doc.get("metrics", {})
                if all(k in m_vals for k in METRICS) and float(m_vals.get("CIDEr", 0)) > 0.5:
                    preds = None
                    if p_file.is_file():
                        preds = json.loads(p_file.read_text(encoding="utf-8")).get("predictions")
                    return m_vals, preds
            except Exception:
                pass
    return None, None

IN_MEMORY_METRICS = defaultdict(dict)  # IN_MEMORY_METRICS[seed][variant] = {metric: score_x100}

def run_or_load_variant_for_seed(variant_name, seed, ckpt_path, cached_ret_for_variant):
    folder_name = FOLDERS_MAP[variant_name]
    var_dir = ABLATION_CACHE_DIR / folder_name / f"seed{seed}"
    var_dir.mkdir(parents=True, exist_ok=True)
    summary_file = var_dir / f"summary_{eval_limit}img.json"
    preds_file   = var_dir / f"predictions_{eval_limit}img.json"
    v_cfg = VARIANT_CONFIGS[variant_name]

    # 1. Kiểm tra trong kết quả đã quét toàn bộ Drive (DRIVE_DISCOVERED_SUMMARIES) hoặc file trực tiếp
    if REUSE_SAVED_PREDICTIONS and not FORCE_RECOMPUTE_METRICS:
        if seed in DRIVE_DISCOVERED_SUMMARIES.get(variant_name, {}):
            m_disc = DRIVE_DISCOVERED_SUMMARIES[variant_name][seed]
            print(f"♻️ [Seed {seed} | {variant_name}] Tái sử dụng ngay kết quả đã lưu trên Drive (CIDEr = {m_disc['CIDEr']*100:.2f}, SPICE = {m_disc.get('SPICE', 0)*100:.2f})!")
            save_json_atomic(summary_file, {
                "seed": seed, "variant": variant_name, "folder": folder_name,
                "protocol": v_cfg["protocol_name"], "n_images": eval_limit,
                "metrics_raw": m_disc,
                "metrics_x100": {k: float(m_disc[k]) * 100.0 for k in METRICS if k in m_disc},
            })
            IN_MEMORY_METRICS[seed][variant_name] = {k: float(m_disc[k]) * 100.0 for k in METRICS if k in m_disc}
            return m_disc

        saved_sum = load_json_with_fallback(summary_file)
        if saved_sum is not None and (
            saved_sum.get("protocol") == v_cfg["protocol_name"]
            and saved_sum.get("n_images") == eval_limit
            and all(m in saved_sum.get("metrics_raw", {}) for m in METRICS)
        ):
            m_raw = saved_sum["metrics_raw"]
            print(f"♻️ [Seed {seed} | {variant_name}] Tái sử dụng kết quả đã hoàn tất từ: {summary_file}")
            IN_MEMORY_METRICS[seed][variant_name] = {k: float(m_raw[k]) * 100.0 for k in METRICS if k in m_raw}
            return m_raw

    # 2. Nếu là nhánh 'full', 'k_5', hoặc 'm_10', tự động tái sử dụng ngay kết quả Đối chứng Level 5 (KB V2) có sẵn trên Drive!
    if variant_name in ("full", "k_5", "m_10") and REUSE_SAVED_PREDICTIONS:
        m_existing, p_existing = try_load_existing_full_l5_kb_v2(seed)
        if m_existing is not None:
            print(f"♻️ [Seed {seed} | {variant_name}] Tái sử dụng ngay kết quả Chuẩn Level 5 (KB V2, K=5, M=10) có sẵn trên Drive (CIDEr = {m_existing['CIDEr']*100:.2f}, SPICE = {m_existing.get('SPICE', 0)*100:.2f})!")
            save_json_atomic(summary_file, {
                "seed": seed, "variant": variant_name, "folder": folder_name,
                "protocol": v_cfg["protocol_name"], "n_images": eval_limit,
                "metrics_raw": m_existing,
                "metrics_x100": {k: float(m_existing[k]) * 100.0 for k in METRICS if k in m_existing},
            })
            IN_MEMORY_METRICS[seed][variant_name] = {k: float(m_existing[k]) * 100.0 for k in METRICS if k in m_existing}
            DRIVE_DISCOVERED_SUMMARIES[variant_name][seed] = m_existing
            return m_existing

    # 2b. Nếu là nhánh 'm_0' (M=0 bộ ba RelTR), tự động tái sử dụng ngay kết quả của 'no_reltr_reranking' trên Drive!
    if variant_name == "m_0" and REUSE_SAVED_PREDICTIONS and seed in DRIVE_DISCOVERED_SUMMARIES.get("no_reltr_reranking", {}):
        m_rel0 = DRIVE_DISCOVERED_SUMMARIES["no_reltr_reranking"][seed]
        print(f"♻️ [Seed {seed} | m_0] Tái sử dụng ngay kết quả M=0 ('no_reltr_reranking') có sẵn trên Drive (CIDEr = {m_rel0['CIDEr']*100:.2f}, SPICE = {m_rel0.get('SPICE', 0)*100:.2f})!")
        save_json_atomic(summary_file, {
            "seed": seed, "variant": "m_0", "folder": folder_name,
            "protocol": v_cfg["protocol_name"], "n_images": eval_limit,
            "metrics_raw": m_rel0,
            "metrics_x100": {k: float(m_rel0[k]) * 100.0 for k in METRICS if k in m_rel0},
        })
        IN_MEMORY_METRICS[seed]["m_0"] = {k: float(m_rel0[k]) * 100.0 for k in METRICS if k in m_rel0}
        DRIVE_DISCOVERED_SUMMARIES["m_0"][seed] = m_rel0
        return m_rel0

    # 3. Kiểm tra cache predictions của nhánh hiện tại (từ DRIVE_DISCOVERED_PREDS hoặc preds_file)
    predictions = []
    if REUSE_SAVED_PREDICTIONS:
        if seed in DRIVE_DISCOVERED_PREDS.get(variant_name, {}):
            p_list = DRIVE_DISCOVERED_PREDS[variant_name][seed]
            if len(p_list) == eval_limit:
                predictions = p_list
                print(f"♻️ [Seed {seed} | {variant_name}] Tái sử dụng {len(predictions):,} câu dự đoán đã lưu sẵn trên Drive (không cần chạy lại Beam Search!).")
            elif 0 < len(p_list) < eval_limit:
                predictions = p_list
                print(f"♻️ [Seed {seed} | {variant_name}] Khôi phục {len(predictions):,}/{eval_limit:,} câu dự đoán dở dang trên Drive, chạy tiếp phần còn lại...")
        if not predictions:
            saved_p = load_json_with_fallback(preds_file)
            if saved_p is not None and saved_p.get("protocol") == v_cfg["protocol_name"]:
                p_list = saved_p.get("predictions", [])
                if len(p_list) == eval_limit:
                    predictions = p_list
                    print(f"♻️ [Seed {seed} | {variant_name}] Tái sử dụng {len(predictions):,} câu dự đoán từ cache.")
                elif 0 < len(p_list) < eval_limit:
                    predictions = p_list
                    print(f"♻️ [Seed {seed} | {variant_name}] Khôi phục {len(predictions):,}/{eval_limit:,} câu dự đoán dở dang, chạy tiếp...")

    if len(predictions) < eval_limit:
        torch.manual_seed(int(seed))
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(int(seed))
        model, _ = load_level5_model_by_seed(seed, ckpt_path)
        t0 = time.time()
        start_idx = len(predictions)
        pbar = tqdm(total=eval_limit, initial=start_idx, desc=f"[{variant_name}] Beam Search GPU (Seed {seed})")
        for b_start in range(start_idx, eval_limit, EVAL_BATCH_SIZE):
            b_end = min(eval_limit, b_start + EVAL_BATCH_SIZE)
            batch_rows = generate_caption_beam_batch_for_variant(
                model,
                test_features[b_start:b_end],
                cached_ret_for_variant[b_start:b_end],
                eval_ids[b_start:b_end],
                variant_name=variant_name,
                primary_seed=seed,
                max_len=MAX_LEN,
                alpha=BEAM_ALPHA,
            )
            predictions.extend(batch_rows)
            pbar.update(len(batch_rows))
            if len(predictions) % 500 == 0 or len(predictions) == eval_limit:
                save_json_atomic(preds_file, {
                    "seed": seed,
                    "variant": variant_name,
                    "protocol": v_cfg["protocol_name"],
                    "n_images": eval_limit,
                    "complete": len(predictions) == eval_limit,
                    "predictions": predictions,
                })
        pbar.close()
        print(f"✅ [Seed {seed} | {variant_name}] Sinh xong {eval_limit:,} câu trong {time.time()-t0:.1f}s.")
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print(f"📊 Đang chấm bộ chỉ số COCO cho [Seed {seed} | {variant_name}]...")
    metrics_raw = evaluate_predictions_coco(predictions)
    save_json_atomic(summary_file, {
        "seed": seed,
        "variant": variant_name,
        "folder": folder_name,
        "protocol": v_cfg["protocol_name"],
        "n_images": eval_limit,
        "metrics_raw": metrics_raw,
        "metrics_x100": {k: float(metrics_raw[k]) * 100.0 for k in METRICS if k in metrics_raw},
    })
    IN_MEMORY_METRICS[seed][variant_name] = {k: float(metrics_raw[k]) * 100.0 for k in METRICS if k in metrics_raw}
    DRIVE_DISCOVERED_SUMMARIES[variant_name][seed] = metrics_raw
    return metrics_raw

# ==============================================================================
# CHẠY ĐỐI CHỨNG 'full' + NHÁNH BÓC TÁCH 'no_mmr' QUA 3 SEEDS (42, 44, 46)
# ==============================================================================
RETRIEVAL_CACHE_BY_VARIANT = {}

for v_name in ["full"] + [v for v in VARIANTS_TO_RUN if v != "full"]:
    need_retrieval = False
    for s_id in CHECKPOINT_PATHS:
        if v_name in ("full", "k_5", "m_10"):
            m_ex, _ = try_load_existing_full_l5_kb_v2(s_id)
            if m_ex is not None:
                continue
        if v_name == "m_0" and s_id in DRIVE_DISCOVERED_SUMMARIES.get("no_reltr_reranking", {}):
            continue
        if s_id in DRIVE_DISCOVERED_SUMMARIES.get(v_name, {}):
            continue
        if len(DRIVE_DISCOVERED_PREDS.get(v_name, {}).get(s_id, [])) == eval_limit:
            continue
        need_retrieval = True
        break
    if need_retrieval:
        k_req = int(VARIANT_CONFIGS[v_name].get("top_k_override", 5))
        print(f"\n🔍 Đang truy hồi Top-{k_req} Tri thức từ Kho V2 cho nhánh '{v_name}' ({FOLDERS_MAP[v_name]})...")
        t_r = time.time()
        RETRIEVAL_CACHE_BY_VARIANT[v_name] = retrieve_knowledge_batch_v2_for_variant(test_features[:eval_limit], eval_ids, v_name, top_k=k_req)
        print(f"✅ Hoàn tất truy hồi nhánh '{v_name}' cho {eval_limit:,} ảnh trong {time.time()-t_r:.1f}s.")

for s_id, ckpt_p in CHECKPOINT_PATHS.items():
    print("\n" + "=" * 78)
    print(f"▶ ĐANG XỬ LÝ SEED {s_id} | ĐỐI CHỨNG 'full' & NHÁNH BÓC TÁCH: {VARIANTS_TO_RUN}")
    print("=" * 78)
    run_or_load_variant_for_seed("full", s_id, ckpt_p, RETRIEVAL_CACHE_BY_VARIANT.get("full"))
    for v_name in VARIANTS_TO_RUN:
        run_or_load_variant_for_seed(v_name, s_id, ckpt_p, RETRIEVAL_CACHE_BY_VARIANT.get(v_name))

# ==============================================================================
# TỔNG HỢP BẢNG KẾT QUẢ TRỰC TIẾP TỪ BỘ NHỚ RAM + TOÀN BỘ FILE TRÊN DRIVE
# ==============================================================================
scan_all_saved_ablation_files_on_drive()
ALL_ORDERED_VARIANTS = ["full"] + [v for v in VARIANTS_TO_RUN if v != "full"]
available_variants_per_seed = defaultdict(dict)

for s_id in CHECKPOINT_PATHS:
    for v_name in ALL_ORDERED_VARIANTS:
        if v_name in IN_MEMORY_METRICS.get(s_id, {}):
            available_variants_per_seed[s_id][v_name] = IN_MEMORY_METRICS[s_id][v_name]
        elif s_id in DRIVE_DISCOVERED_SUMMARIES.get(v_name, {}):
            m_raw = DRIVE_DISCOVERED_SUMMARIES[v_name][s_id]
            available_variants_per_seed[s_id][v_name] = {k: float(m_raw[k]) * 100.0 for k in METRICS}

per_seed_comparison_rows = {}
for s_id in CHECKPOINT_PATHS:
    if "full" in available_variants_per_seed[s_id]:
        rows_s = []
        full_m = available_variants_per_seed[s_id]["full"]
        for v_name in ALL_ORDERED_VARIANTS:
            if v_name in available_variants_per_seed[s_id]:
                m_x100 = available_variants_per_seed[s_id][v_name]
                row = {"seed": s_id, "variant": v_name, "folder": FOLDERS_MAP[v_name], **m_x100}
                row.update({f"delta_{m}": m_x100[m] - full_m[m] for m in METRICS})
                rows_s.append(row)
        per_seed_comparison_rows[s_id] = rows_s

        try:
            seed_base_dir = ABLATION_CACHE_DIR / "_baseline" / f"seed{s_id}"
            seed_base_dir.mkdir(parents=True, exist_ok=True)
            csv_path = seed_base_dir / "comparison.csv"
            with csv_path.open("w", encoding="utf-8-sig", newline="") as f:
                writer = csv.DictWriter(f, fieldnames=["seed", "variant", "folder"] + METRICS + [f"delta_{m}" for m in METRICS])
                writer.writeheader()
                writer.writerows(rows_s)
            save_json_atomic(seed_base_dir / "comparison.json", {
                "seed": s_id,
                "n_images": eval_limit,
                "kb_version": "KB_V2_knowledge_base_level4",
                "rows": rows_s,
            })
        except Exception as e:
            print(f"ℹ️ Bỏ qua lỗi ghi file phụ comparison.csv ({e}), vẫn hiển thị đầy đủ bảng kết quả bên dưới.")

# 1. Hiển thị bảng DataFrame cho từng Seed
for s_id, rows_s in per_seed_comparison_rows.items():
    print("\n" + "=" * 92)
    print(f"📊 BẢNG KẾT QUẢ BÓC TÁCH TRÊN KHO MỚI V2 — SEED {s_id} ({eval_limit:,} ẢNH TEST)")
    print(" (Đơn vị: Metric × 100 | delta = Bỏ thành phần − Đầy đủ; Số âm nghĩa là bỏ thành phần làm giảm điểm)")
    print("=" * 92)
    df_s = pd.DataFrame(rows_s)
    display(df_s[["variant", "folder"] + METRICS].round(4))
    display(df_s[["variant"] + [f"delta_{m}" for m in METRICS]].round(4))

# 2. Hiển thị Bảng Tổng hợp Trung bình Đa Seed (Mean ± SD) cho MỌI NHÁNH có dữ liệu
variants_with_any_seed = [v for v in ALL_ORDERED_VARIANTS if any(v in available_variants_per_seed[s] for s in CHECKPOINT_PATHS)]
if "full" in variants_with_any_seed:
    mean_rows = []
    full_seeds = [s for s in CHECKPOINT_PATHS if "full" in available_variants_per_seed[s]]
    full_means = {
        m: statistics.mean([available_variants_per_seed[s]["full"][m] for s in full_seeds])
        for m in METRICS
    }
    for v_name in variants_with_any_seed:
        v_seeds = [s for s in CHECKPOINT_PATHS if v_name in available_variants_per_seed[s]]
        r_mean = {"variant": v_name, "folder": FOLDERS_MAP[v_name], "seeds": str(v_seeds)}
        for m in METRICS:
            vals = [available_variants_per_seed[s][v_name][m] for s in v_seeds]
            mu = statistics.mean(vals)
            sd = statistics.stdev(vals) if len(vals) > 1 else 0.0
            r_mean[m] = f"{mu:.2f} ± {sd:.2f}"
            r_mean[f"delta_{m}"] = round(mu - full_means[m], 2)
        mean_rows.append(r_mean)

    print("\n" + "=" * 108)
    print(f"🏆 BẢNG TỔNG HỢP TRUNG BÌNH ĐA SEED ({list(CHECKPOINT_PATHS.keys())}) — NGHIÊN CỨU BÓC TÁCH TRÊN KHO MỚI V2 (COCO 2014)")
    print("=" * 108)
    df_mean = pd.DataFrame(mean_rows)
    display(df_mean[["variant", "folder", "seeds"] + METRICS])
    display(df_mean[["variant", "seeds"] + [f"delta_{m}" for m in METRICS]])

    OLD_V1_DELTAS = {
        "full":                  {"CIDEr_V1": 110.92, "delta_CIDEr_V1": 0.00},
        "k_1":                   {"CIDEr_V1": 108.59, "delta_CIDEr_V1": -2.25},
        "k_3":                   {"CIDEr_V1": 110.56, "delta_CIDEr_V1": -0.28},
        "k_5":                   {"CIDEr_V1": 110.84, "delta_CIDEr_V1": 0.00},
        "k_10":                  {"CIDEr_V1": 110.85, "delta_CIDEr_V1": +0.01},
        "k_20":                  {"CIDEr_V1": 110.90, "delta_CIDEr_V1": +0.06},
        "m_0":                   {"CIDEr_V1": 105.70, "delta_CIDEr_V1": -5.93},
        "m_3":                   {"CIDEr_V1": 108.92, "delta_CIDEr_V1": -2.71},
        "m_5":                   {"CIDEr_V1": 110.45, "delta_CIDEr_V1": -1.18},
        "m_10":                  {"CIDEr_V1": 111.63, "delta_CIDEr_V1": 0.00},
        "m_20":                  {"CIDEr_V1": 111.40, "delta_CIDEr_V1": -0.23},
    }
    print("\n" + "=" * 108)
    print("🔬 ĐỐI CHIẾU HIỆU QUẢ BÓC TÁCH: KHO TRI THỨC CŨ (V1) vs KHO TRI THỨC MỚI (V2 — SPO PROTOTYPE)")
    print("=" * 108)
    print(f" {'Nhánh Bóc tách (Variant)':<26} | {'CIDEr Kho Cũ V1 (Δ)':<24} | {'CIDEr Kho Mới V2 (Mean ± SD)':<28} | {'Δ CIDEr Kho Mới V2':>20}")
    print("-" * 108)
    for r_m in mean_rows:
        vn = r_m["variant"]
        v1_info = OLD_V1_DELTAS.get(vn, {})
        v1_str = f"{v1_info.get('CIDEr_V1', 0):.2f} ({v1_info.get('delta_CIDEr_V1', 0):+.2f})"
        v2_str = r_m["CIDEr"]
        d_v2 = r_m["delta_CIDEr"]
        print(f" {vn:<26} | {v1_str:<24} | {v2_str:<28} | {d_v2:+19.2f}")
    print("=" * 108)

    try:
        master_csv = ABLATION_CACHE_DIR / "comparison_multiseed_kb_v2.csv"
        df_mean.to_csv(master_csv, index=False, encoding="utf-8-sig")
        print(f"✅ Đã lưu bảng tổng hợp Bóc tách Đa Seed (Kho Mới V2) tại: {master_csv}")
    except Exception:
        pass


🔎 Tìm thấy sẵn kết quả hoàn tất trên Drive cho nhánh 'k_10': Seeds [42]
🔎 Tìm thấy sẵn kết quả hoàn tất trên Drive cho nhánh 'k_3': Seeds [42]
🔎 Tìm thấy sẵn kết quả hoàn tất trên Drive cho nhánh 'm_20': Seeds [42, 44]
🔎 Tìm thấy sẵn kết quả hoàn tất trên Drive cho nhánh 'k_1': Seeds [42]
🔎 Tìm thấy sẵn kết quả hoàn tất trên Drive cho nhánh 'm_5': Seeds [42, 44]
🔎 Tìm thấy sẵn kết quả hoàn tất trên Drive cho nhánh 'k_5': Seeds [42]
🔎 Tìm thấy sẵn kết quả hoàn tất trên Drive cho nhánh 'm_3': Seeds [42, 44]
🔎 Tìm thấy sẵn kết quả hoàn tất trên Drive cho nhánh 'm_10': Seeds [42, 44, 46]
🔎 Tìm thấy sẵn kết quả hoàn tất trên Drive cho nhánh 'm_0': Seeds [42, 44, 46]
🔎 Tìm thấy sẵn kết quả hoàn tất trên Drive cho nhánh 'no_gate_only': Seeds [42, 44, 46]
🔎 Tìm thấy sẵn kết quả hoàn tất trên Drive cho nhánh 'no_multitoken_only': Seeds [42, 44, 46]
🔎 Tìm thấy sẵn kết quả hoàn tất trên Drive cho nhánh 'no_reltr_reranking': Seeds [42, 44, 46]
🔎 Tìm thấy sẵn kết quả hoàn tất trên Drive cho nhánh '

[k_20] Beam Search GPU (Seed 42):  90%|######### | 4500/5000 [00:00<?, ?it/s]

✅ Đã nạp strict=True thành công Level 5 (Seed 42) | State SHA-256: c1f34c519e65f348...
✅ Đã nạp strict=True thành công Level 5 (Seed 44) | State SHA-256: b441c261e50f4aec...
✅ Đã nạp strict=True thành công Level 5 (Seed 46) | State SHA-256: 8bdeed0b5e8eeebb...
✅ [Seed 42 | k_20] Sinh xong 5,000 câu trong 29.3s.
📊 Đang chấm bộ chỉ số COCO cho [Seed 42 | k_20]...
{'testlen': 47915, 'reflen': 47522, 'guess': [47915, 42915, 37915, 32915], 'correct': [35783, 19583, 10156, 5320]}
ratio: 1.0082698539623542
  + Bleu_1    :  74.68
  + Bleu_2    :  58.38
  + Bleu_3    :  45.03
  + Bleu_4    :  34.85
  + METEOR    :  27.79
  + ROUGE_L   :  56.09
  + CIDEr     : 114.79
  ⏳ Đang chấm SPICE (-Xmx4G & CoreNLP cache)...
  ✅ SPICE     :  20.66 (1065.1s)
  + CHAIRs    :   8.28% | CHAIRi:   7.23%

▶ ĐANG XỬ LÝ SEED 44 | ĐỐI CHỨNG 'full' & NHÁNH BÓC TÁCH: ['k_5', 'k_1', 'k_3', 'k_10', 'k_20']
♻️ [Seed 44 | full] Tái sử dụng ngay kết quả đã lưu trên Drive (CIDEr = 115.36, SPICE = 20.73)!
♻️ [Seed 44 | k_5]

[k_1] Beam Search GPU (Seed 44):   0%|          | 0/5000 [00:00<?, ?it/s]

✅ [Seed 44 | k_1] Sinh xong 5,000 câu trong 209.0s.
📊 Đang chấm bộ chỉ số COCO cho [Seed 44 | k_1]...
{'testlen': 48017, 'reflen': 47550, 'guess': [48017, 43017, 38017, 33017], 'correct': [35622, 19352, 9886, 5108]}
ratio: 1.0098212407991376
  + Bleu_1    :  74.19
  + Bleu_2    :  57.77
  + Bleu_3    :  44.27
  + Bleu_4    :  34.04
  + METEOR    :  27.46
  + ROUGE_L   :  55.62
  + CIDEr     : 112.15
  ⏳ Đang chấm SPICE (-Xmx4G & CoreNLP cache)...
  ✅ SPICE     :  20.39 (88.3s)
  + CHAIRs    :   8.94% | CHAIRi:   7.78%
✅ Đã nạp strict=True thành công Level 5 (Seed 44) | State SHA-256: b441c261e50f4aec...


[k_3] Beam Search GPU (Seed 44):   0%|          | 0/5000 [00:00<?, ?it/s]

... (21 outputs left out)


🏆 BẢNG TỔNG HỢP TRUNG BÌNH ĐA SEED ([42, 44, 46]) — NGHIÊN CỨU BÓC TÁCH TRÊN KHO MỚI V2 (COCO 2014)


,variant,folder,seeds,Bleu_1,Bleu_2,Bleu_3,Bleu_4,METEOR,ROUGE_L,CIDEr,SPICE
0,full,_baseline,"[42, 44, 46]",74.92 ± 0.05,58.57 ± 0.03,45.13 ± 0.04,34.90 ± 0.06,27.85 ± 0.02,56.01 ± 0.01,115.41 ± 0.07,20.76 ± 0.02
1,k_5,TopK_K5_Chuan,"[42, 44, 46]",74.92 ± 0.05,58.57 ± 0.03,45.13 ± 0.04,34.90 ± 0.06,27.85 ± 0.02,56.01 ± 0.01,115.41 ± 0.07,20.76 ± 0.02
2,k_1,TopK_K1,"[42, 44, 46]",74.21 ± 0.04,57.79 ± 0.04,44.32 ± 0.04,34.11 ± 0.06,27.46 ± 0.04,55.61 ± 0.07,112.13 ± 0.04,20.38 ± 0.04
3,k_3,TopK_K3,"[42, 44, 46]",74.73 ± 0.03,58.42 ± 0.03,45.05 ± 0.02,34.85 ± 0.03,27.84 ± 0.01,55.99 ± 0.01,114.76 ± 0.13,20.66 ± 0.03
4,k_10,TopK_K10,"[42, 44, 46]",74.83 ± 0.05,58.53 ± 0.06,45.18 ± 0.05,35.00 ± 0.03,27.84 ± 0.03,56.09 ± 0.03,115.16 ± 0.17,20.73 ± 0.02
5,k_20,TopK_K20,"[42, 44, 46]",74.64 ± 0.03,58.33 ± 0.04,44.98 ± 0.05,34.83 ± 0.04,27.79 ± 0.00,56.07 ± 0.02,114.81 ± 0.09,20.65 ± 0.02


,variant,seeds,delta_Bleu_1,delta_Bleu_2,delta_Bleu_3,delta_Bleu_4,delta_METEOR,delta_ROUGE_L,delta_CIDEr,delta_SPICE
0,full,"[42, 44, 46]",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
1,k_5,"[42, 44, 46]",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2,k_1,"[42, 44, 46]",-0.71,-0.79,-0.82,-0.79,-0.39,-0.40,-3.28,-0.38
3,k_3,"[42, 44, 46]",-0.19,-0.15,-0.09,-0.05,-0.01,-0.03,-0.66,-0.09
4,k_10,"[42, 44, 46]",-0.08,-0.04,0.05,0.10,-0.00,0.07,-0.26,-0.03
5,k_20,"[42, 44, 46]",-0.27,-0.24,-0.15,-0.07,-0.05,0.06,-0.61,-0.11



🔬 ĐỐI CHIẾU HIỆU QUẢ BÓC TÁCH: KHO TRI THỨC CŨ (V1) vs KHO TRI THỨC MỚI (V2 — SPO PROTOTYPE)
 Nhánh Bóc tách (Variant)   | CIDEr Kho Cũ V1 (Δ)      | CIDEr Kho Mới V2 (Mean ± SD) |   Δ CIDEr Kho Mới V2
------------------------------------------------------------------------------------------------------------
 full                       | 110.92 (+0.00)           | 115.41 ± 0.07                |               +0.00
 k_5                        | 110.84 (+0.00)           | 115.41 ± 0.07                |               +0.00
 k_1                        | 108.59 (-2.25)           | 112.13 ± 0.04                |               -3.28
 k_3                        | 110.56 (-0.28)           | 114.76 ± 0.13                |               -0.66
 k_10                       | 110.85 (+0.01)           | 115.16 ± 0.17                |               -0.26
 k_20                       | 110.90 (+0.06)           | 114.81 ± 0.09                |               -0.61
✅ Đã lưu bảng tổng hợp Bóc tách Đa Seed 